In [ ]:
import numpy as np
import xarray as xr
from pathlib import Path
from itertools import count
from string import ascii_lowercase as alc
import scipy.signal
from scipy import stats
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize, TwoSlopeNorm, ListedColormap
from matplotlib.ticker import AutoMinorLocator
from matplotlib.offsetbox import AnchoredText
from matplotlib.collections import LineCollection
from mpl_toolkits.axes_grid1 import make_axes_locatable
import cartopy.crs as ccrs
import cmcrameri as scm

import mw_protocol.spreading as spreading
import mw_protocol.plotting as plotting
import mw_protocol.saving as saving
import pylaeoclim_leeds.util_hadcm3 as util

In [ ]:
# To modify with your data folder. !This does not include GCM outputs.
home_folder = str(Path.home())
script_folder = str(Path.cwd())

data_folder = f"{script_folder}/data"
ice_sheet_folder = f"{data_folder}/ice_sheet_reconstructions"
output_folder = f"{script_folder}/outputs"

# Suplementary Information - Last deglaciation simulations sensitivity to the choice of ice sheets and meltwater discharge

In [ ]:
database = {'tecac':f"{data_folder}/ice6g_hs1/lgm_control", 
            'xoupa':f"{data_folder}/database/xoupa", 
            'xpfjb':f"{data_folder}/database/xpfjb",
            'xpfjc':f"{data_folder}/database/xpfjc", 
            'xpfjd':f"{data_folder}/database/xpfjd", 
            'xpfje':f"{data_folder}/database/xpfje",
            'xqctb':f"{data_folder}/database/xqctb",
            'xqctc':f"{data_folder}/database/xqctc"}

In [ ]:
color_expt = {'xoupa':'#232E40', 'tecac':'#4A222E',
              'xpfjb':'#8C2E2E', 'xpfjc':'#AD5956',
              'xpfjd':'#004266', 'xpfje':'#6BABC1', 
              'xqctb':'xkcd:ocean green', 'xqctc':"#B48A18"}

start_expt = {'tecac':2552+22000, 'xoupa':22000, 'xpfjb':100000, 'xpfjc':100000, 'xpfjd':100000, 'xpfje':100000, 
              'xqctb':100000, 'xqctc':100000}

name_expt = {'tecac':"ICE6G_noMw", 'xoupa':"GLAC1D_noMw",
              'xpfjb':"ICE6G_mw", 'xpfjc':"ICE6G_GLAC1DMw",
              'xpfjd':"GLAC1D_mw", 'xpfje':"GLAC1D_ICE6GMw",
              'xqctc':"GLAC1D_ts"}

ice_expt = {'tecac':"ICE6G", 'xoupa':"GLAC-1D",
              'xpfjb':"ICE6G", 'xpfjc':"ICE6G",
              'xpfjd':"GLAC-1D", 'xpfje':"GLAC-1D",
            'xqctb':"GLAC-1D", 'xqctc':"GLAC-1D"}

label_expt = {'tecac':"ICE6G control", 'xoupa':"GLAC-1D control",
              'xpfjb':"ICE6G ice-sheet, ICE6G meltwater", 'xpfjc':"ICE6G ice-sheet, GLAC-1D meltwater",
              'xpfjd':"GLAC-1D ice-sheet, GLAC-1D meltwater", 'xpfje':"GLAC-1D ice-sheet, ICE6G meltwater",
              'xqctb':"GLAC-1D ghg transient", 'xqctc':"GLAC-1D full transient"}

In [ ]:
color_fluxes = {'elwg': 'xkcd:deep blue', 'gin':'xkcd:dark pink', 'med':'xkcd:avocado green',
                'arc':'xkcd:kelley green', 'so':'xkcd:lightblue', 'pac':'xkcd:sandy', 'tot':'black'}
label_fluxes = {'elwg':'East Laurentide and\nWest Greenland', 'gin':'GIN seas', 'med':'Mediterranean Sea',
                'arc':'Arctic', 'so':'Southern Ocean', 'pac':'Pacific', 'tot':'Total'}

In [ ]:
ds_basin = xr.open_dataset(f"{data_folder}/basin_hadcm3_glac1d_lgm.nc")

masks = {}

masks['arc'] = ds_basin.arctic
masks['na'] = ds_basin.atlantic.sel(latitude=slice(44,79))
masks['tpa'] = ds_basin.atlantic.sel(latitude=slice(-33,44))
masks['sa'] = ds_basin.atlantic.sel(latitude=slice(-60,-33))
masks['pac'] = ds_basin.pacific
masks['so'] = ds_basin.southern
masks['idn'] = ds_basin.indian

color_zones = {'arc':'#A4D4D2',
               'na':'#00316A', 'ena':'xkcd:blue green', 'wna':'xkcd:navy', 
               'tpa':'#306B34', 
               'sa':'#FEE000', 
               'pac':'#B41001', 'npc':'xkcd:light red', 'spc':'xkcd:dark red',
               'idn':'#F68307', 
               'soa':'xkcd:wisteria', 'soi':'xkcd:pinky purple', 'sop':'xkcd:royal purple', 
               'so':'#953878', 
               'tot':'xkcd:black', 
               'gin':'#DFBA47', 'irm':'#CFABA0', 'ls':'#CC573D', 
               'spg':'#C4BE81', 'eur':'#9EA550'}


label_zones = {'arc':'Arctic', 
               'ena':'East North Atlantic', 'wna':'West North Atlantic', 
               'na': 'North Atlantic', 'tpa':'Subtropical Atlantic', 'sa':'South Atlantic',
               'pac':'Pacific', 'npc':'North Pacific', 'spc':'South Pacific',
               'idn':'Indian',
               'so':'Southern', 'soa':'SO Atlantic', 'soi':'SO Indian', 'sop':'SO Pacific', 
               'tot':'Total',
               'gin':'GIN seas', 'irm':'Irminger Sea', 'ls':'Labrador Sea', 
               'spg':'Subpolar gyre', 'eur':'European coast'}


color_zones['nor'] = color_zones['gin']
label_zones['nor'] = "Norwegian Sea"
color_zones['icd'] = color_zones['eur']
label_zones['icd'] = "Iceland Basin"

masks_na = {}

masks_na['gin'] = xr.concat([ds_basin.atlantic.sel(longitude=slice(340,360)).sel(latitude=slice(64,79)),
                          ds_basin.atlantic.sel(longitude=slice(0,20)).sel(latitude=slice(64,79))], 
                         dim='longitude')
masks_na['eur'] = xr.concat([ds_basin.atlantic.sel(longitude=slice(340,360)).sel(latitude=slice(44,64)),
                          ds_basin.atlantic.sel(longitude=slice(0,2)).sel(latitude=slice(44,64))], 
                         dim='longitude')
masks_na['irm'] = ds_basin.atlantic.sel(longitude=slice(316,339)).sel(latitude=slice(53,70))
masks_na['ls'] = ds_basin.atlantic.sel(longitude=slice(280,315)).sel(latitude=slice(53,79))
masks_na['spg'] = ds_basin.atlantic.sel(longitude=slice(280,339)).sel(latitude=slice(44,53))
masks_na['arc'] = ds_basin.arctic


In [ ]:
ds_basin_atm = xr.open_dataset(f"{data_folder}/basin_hadcm3_glac1d_lgm_atm.nc")

masks_na_atm = {}

masks_na_atm['gin'] = xr.concat([ds_basin_atm.atlantic.sel(longitude=slice(339,360)).sel(latitude=slice(64,79)),
                          ds_basin_atm.atlantic.sel(longitude=slice(0,20)).sel(latitude=slice(64,79))], 
                         dim='longitude')
masks_na_atm['eur'] = xr.concat([ds_basin_atm.atlantic.sel(longitude=slice(339,360)).sel(latitude=slice(44,64)),
                          ds_basin_atm.atlantic.sel(longitude=slice(0,3)).sel(latitude=slice(44,64))], 
                         dim='longitude')
masks_na_atm['irm'] = ds_basin_atm.atlantic.sel(longitude=slice(316,339)).sel(latitude=slice(53,70))
masks_na_atm['ls'] = ds_basin_atm.atlantic.sel(longitude=slice(279,315)).sel(latitude=slice(53,79))
masks_na_atm['spg'] = ds_basin_atm.atlantic.sel(longitude=slice(280,339)).sel(latitude=slice(44,53))
masks_na_atm['arc'] = ds_basin_atm.arctic


In [ ]:
color_spans = {'cold':'#2C4251', 
               'warming':'#FFABAB',
               'merid':'#C9381B', 
               'zonal':'#F59F00', 
               'cooling':'#B7DDF6'}

## Figure - Comparison

In [ ]:
snoll23_folder = f"{home_folder}/work/data/snoll_egu_23"

In [ ]:
color_expt['deglh'] = '#AA3D22'
color_expt['iloveclim_ice6g'] = '#815F72'
color_expt['iloveclim_glac1d'] = '#2A9D8F'
color_expt['miroc'] = '#CE5370'
color_expt['mpi_ice6g'] = '#FFBF47'
color_expt['mpi_glac1d'] = "#365659"

name_expt['deglh'] = "HadCM3_ICE6G"
name_expt['iloveclim_ice6g'] = "iLOVECLIM_ICE6G"
name_expt['iloveclim_glac1d'] = "iLOVECLIM_GLAC1D"
name_expt['miroc'] = "MIROC_ICE6G"
name_expt['mpi_ice6g'] = "MPI_ICE6G"
name_expt['mpi_glac1d'] = "MPI_GLAC1D"


In [ ]:
time_sim = {}

time_sim['deglh'] = -np.load(f"{snoll23_folder}/time10_hadcm3.npy")*1000
time_sim['iloveclim_ice6g'] = -np.load(f"{snoll23_folder}/time10_iloveclim.npy")[:900]*1000
time_sim['iloveclim_glac1d'] = -np.load(f"{snoll23_folder}/time10_iloveclim.npy")[:900]*1000
time_sim['miroc'] = -np.load(f"{snoll23_folder}/time10_miroc.npy")*1000
time_sim['mpi_ice6g'] = -np.load(f"{snoll23_folder}/time10_mpi.npy")*1000
time_sim['mpi_glac1d'] = -np.load(f"{snoll23_folder}/time10_mpi.npy")*1000


In [ ]:
mw_flux = {}

mw_flux['deglh'] = np.load(f"{snoll23_folder}/fwf_flux/deglh.freshwater_flux.010yr.npy")
mw_flux['iloveclim_ice6g'] = pd.read_csv(f"{snoll23_folder}/fwf_flux/fwf_ice6gc_iloveclim.csv", usecols=[0,2], names=['time','mw'], header=0)
mw_flux['iloveclim_glac1d'] = pd.read_csv(f"{snoll23_folder}/fwf_flux/fwf_glac_iloveclim.csv", usecols=[0,1], names=['time','mw'], header=0)
mw_flux['miroc'] = pd.read_csv(f"{snoll23_folder}/fwf_flux/miroc_freshwater_flux.csv", names=['time','mw'], header=0)
mw_flux['mpi_ice6g'] = pd.read_csv(f"{snoll23_folder}/fwf_flux/pmu0208_global_meltflux.csv", usecols=[2,3], names=['time','mw'], header=0)
mw_flux['mpi_glac1d'] = pd.read_csv(f"{snoll23_folder}/fwf_flux/pmu0207_global_meltflux.csv", usecols=[2,3], names=['time','mw'], header=0)


In [ ]:
fluxes, lsm = {}, {}
time = {}

# ICE6G

ds_discharge = xr.open_dataset(f"{data_folder}/mw_inputs/xpfj.wfix.ice6g_ts.shift.nc",decode_times=False)
ds_discharge.attrs['waterfix'] = 'GLAC-1D'
ds_lsm = xr.open_dataset(f"{data_folder}/lgm_inputs/ice6g.omask.nc")
ds_wfix = xr.open_dataset(f"{data_folder}/lgm_inputs/teadv3.qrparm.waterfix.nc")

fluxes['ICE6G'] = plotting.create_discharge_ts(plotting.remove_waterfix(
    saving.ancil_to_discharge(ds_discharge), ds_wfix),
                                    ds_lsm, details='low', rmean=1)

lsm['ICE6G'] = ds_lsm.lsm

time['ICE6G'] = ds_discharge.t.values - 100000


# GLAC-1D

ds_discharge = xr.open_dataset(f"{data_folder}/mw_inputs/xoup.wfix.glac_ts.nc",decode_times=False)
ds_discharge.attrs['waterfix'] = 'GLAC-1D'

ds_wfix = xr.open_dataset(f"{data_folder}/lgm_inputs/qrparm.waterfix.hadcm3.nc")
ds_lsm = xr.open_dataset(f"{data_folder}/lgm_inputs/temev.qrparm.omask.nc")

fluxes['GLAC-1D'] = plotting.create_discharge_ts(plotting.remove_waterfix(
    saving.ancil_to_discharge(ds_discharge), ds_wfix),
                                    ds_lsm, details='low', rmean=1)
lsm['GLAC-1D'] = ds_lsm.lsm

time['GLAC-1D'] = ds_discharge.t.values

In [ ]:
flux_dict = {}
for mw in ['GLAC-1D','ICE6G']:
    flux_dict[mw] = {}
    for key in [flux for flux in fluxes[mw].keys() if flux not in ['tot']]:
        flux_dict[mw][key] = fluxes[mw][key].value

mw_flux['hadcm3_ice6g'] = np.sum([flux_dict['ICE6G'][key] for key in flux_dict['ICE6G'].keys()], axis=0)
mw_flux['hadcm3_glac1d'] = np.sum([flux_dict['GLAC-1D'][key] for key in flux_dict['GLAC-1D'].keys()], axis=0)

In [ ]:
amoc_max = {}

for expt in ['xpfjb', 'xpfjc', 'xpfjd', 'xpfje', 'xqctc']:
    amoc_max[expt] = xr.open_dataset(
        f"{database[expt]}/time_series/{expt}.merid.annual.nc").Merid_Atlantic.sel(
        latitude=slice(0,90)).sel(depth=slice(500,3500)).max(['latitude','depth'])

amoc_max['deglh'] = np.load(f"{snoll23_folder}/AMOC/deglh.NH_max500-3500m.merid_Atlantic_ym_dpth.010yr.npy")
amoc_max['iloveclim_ice6g'] = np.load(f"{snoll23_folder}/AMOC/iloveclim_fwf.NH_max500-3500m.merid_Atlantic_ym_dpth.010yr.npy")[:900]
amoc_max['iloveclim_glac1d'] = np.load(f"{snoll23_folder}/AMOC/iloveclim_fwf_glac.NH_max500-3500m.merid_Atlantic_ym_dpth.010yr.npy")[:900]
amoc_max['miroc'] = np.load(f"{snoll23_folder}/AMOC/miroc.NH_max500-3500m.merid_Atlantic_ym_dpth.010yr.npy")
amoc_max['mpi_ice6g'] = np.load(f"{snoll23_folder}/AMOC/pmu0211.NH_max500-3500m.merid_Atlantic_ym_dpth.010yr.npy")
amoc_max['mpi_glac1d'] = np.load(f"{snoll23_folder}/AMOC/pmu0212.NH_max500-3500m.merid_Atlantic_ym_dpth.010yr.npy")


In [ ]:
ngrip = {}

for expt in ['xpfjb', 'xpfjc', 'xpfjd', 'xpfje', 'xqctc']:
    ngrip[expt] = xr.open_dataset(
        f"{database[expt]}/{expt}.temp2m.annual.nc").temp_mm_1_5m.sel(
        longitude=318, method='nearest').sel(latitude=75, method='nearest') - 273.15


ngrip['deglh'] = xr.open_dataset(
    f"{snoll23_folder}/SAT/deglh.temp_mm_1_5m.010yr.nc").temp_mm_1_5m.isel(ht=0).sel(
        longitude=318, method='nearest').sel(latitude=75, method='nearest') - 273.15

ngrip['iloveclim_ice6g'] = xr.open_dataset(
    f"{snoll23_folder}/SAT/iloveclim_fwf.temp_mm_1_5m.010yr.nc").t2m.sel(
        lon=318, method='nearest').sel(lat=75, method='nearest')

ngrip['iloveclim_glac1d'] = xr.open_dataset(
    f"{snoll23_folder}/SAT/iloveclim_fwf_glac.temp_mm_1_5m.010yr.nc").t2m.sel(
        lon=318, method='nearest').sel(lat=75, method='nearest')

ngrip['miroc'] = xr.open_dataset(
    f"{snoll23_folder}/SAT/miroc.temp_mm_1_5m.010yr.nc", decode_times=False).tas.sel(
        lon=318, method='nearest').sel(lat=75, method='nearest')

ngrip['mpi_ice6g'] = xr.open_dataset(
    f"{snoll23_folder}/SAT/pmu0211.temp_mm_1_5m.010yr.nc", decode_times=False).temp2.sel(
        lon=318, method='nearest').sel(lat=75, method='nearest')

ngrip['mpi_glac1d'] = xr.open_dataset(
    f"{snoll23_folder}/SAT/pmu0212.temp_mm_1_5m.010yr.nc", decode_times=False).temp2.sel(
        lon=318, method='nearest').sel(lat=75, method='nearest')


**AMOC - Poeppelmeier et al. 2023**

In [ ]:
ds_pop_pi = xr.open_dataset(f"{data_folder}/poeppelmeier_nature_2023/Poeppelmeier2022_PIctrl.nc")
amoc_pop_pi = ds_pop_pi.AtlanticStreamfunction.sel(lat_u=26.5, method='nearest').sel(z_w=slice(300,5000)).max(dim='z_w')

ds_pop_deglac = xr.open_dataset(f"{data_folder}/poeppelmeier_nature_2023/Poeppelmeier2022_DeglacialBestFit.nc")
amoc_pop_deglac = ds_pop_deglac.AtlanticStreamfunction.sel(lat_u=26.5, method='nearest').sel(z_w=slice(300,5000)).max(dim='z_w')/amoc_pop_pi.values[()]*100

ds_pop_lgm = xr.open_dataset(f"{data_folder}/poeppelmeier_nature_2023/Poeppelmeier2022_LGMBestFit.nc")
amoc_pop_lgm = ds_pop_lgm.AtlanticStreamfunction.sel(lat_u=26.5, method='nearest').sel(z_w=slice(300,5000)).max(dim='z_w')/amoc_pop_pi.values[()]*100

plot_amoc_pop_lgm_time = np.arange(-21_000, amoc_pop_deglac.time.values[0], 100)
plot_amoc_pop_lgm = np.linspace(amoc_pop_lgm.values[()], amoc_pop_deglac.values[0], len(plot_amoc_pop_lgm_time))


**NGRIP - Martin et al. 2023**

In [ ]:
proxy_ngrip = pd.read_excel(f"{data_folder}/martin_nature_2023/41586_2023_5875_MOESM3_ESM.xlsx", sheet_name='Fig 1c',
              skiprows=14, header=0, names=['age', 'age_bic', 'temp', 'source'])


**Plot**

In [ ]:
fig = plt.figure(figsize=(7,12), dpi=300)

gs = gridspec.GridSpec(nrows=8, ncols=1, hspace=0, height_ratios=[1,10,10,10,10,10,10,10])
axNotes = fig.add_subplot(gs[0,0],facecolor='None')
axMW = fig.add_subplot(gs[1,0],facecolor='None')
axAMOChcm3 = fig.add_subplot(gs[2,0], sharex=axMW, facecolor='None')
axAMOCsim = fig.add_subplot(gs[3,0], sharex=axMW, sharey=axAMOChcm3, facecolor='None')
axAMOCpop = fig.add_subplot(gs[4,0], sharex=axMW, facecolor='None')
axNGRIPhcm3 = fig.add_subplot(gs[5,0], sharex=axMW, facecolor='None')
axNGRIPsim = fig.add_subplot(gs[6,0], sharex=axMW, sharey=axNGRIPhcm3, facecolor='None')
axNGRIPproxy = fig.add_subplot(gs[7,0], sharex=axMW, sharey=axNGRIPhcm3, facecolor='None')


## Meltwater Flux

axMW.plot(time['ICE6G'], mw_flux['hadcm3_ice6g'], color=color_expt['xpfjb'], label=name_expt['xpfjb'], lw=1.5, zorder=1)
axMW.plot(time['GLAC-1D'], mw_flux['hadcm3_glac1d'], color=color_expt['xpfjd'], label=name_expt['xpfjd'], lw=1.5, zorder=1)
# axMW.plot(time['deglh'], mw_flux['deglh'], color=color_expt['deglh'], label=name_expt['deglh'])
for expt in ['iloveclim_ice6g', 'iloveclim_glac1d', 'miroc', 'mpi_ice6g', 'mpi_glac1d']:
    axMW.plot(-mw_flux[expt].time*1000, mw_flux[expt].mw, color=color_expt[expt], label=name_expt[expt], lw=1, zorder=0)

    
## AMOC simulations
for expt in ['xpfjb', 'xpfjd', 'xqctc']:
    axAMOChcm3.plot(amoc_max[expt].t.dt.year-start_expt[expt], util.rmean(amoc_max[expt].values, 30), 
               color=color_expt[expt], linestyle="-", label=name_expt[expt], lw=1)
        
for expt in ['deglh', 'iloveclim_ice6g', 'iloveclim_glac1d', 'miroc', 'mpi_ice6g', 'mpi_glac1d']:
    axAMOCsim.plot(time_sim[expt], util.rmean(amoc_max[expt], 3), 
                 color=color_expt[expt], linestyle="-", label=name_expt[expt], lw=1)

# AMOC Proxies

axAMOCpop.plot(plot_amoc_pop_lgm_time, plot_amoc_pop_lgm, 
               color='black', label="Poeppelmeier 2023 (LGM interpolation)", lw=1, ls='--')
axAMOCpop.plot(amoc_pop_deglac.time, amoc_pop_deglac, 
               color='black', label="Poeppelmeier 2023 (Deglacial)", lw=2)

    
## NGRIP Simulations
for expt in ['xpfjb', 'xpfjd', 'xqctc']:
    axNGRIPhcm3.plot(ngrip[expt].t.dt.year-start_expt[expt], util.rmean(ngrip[expt].values, 30) - util.rmean(ngrip[expt].values, 30)[0], 
               color=color_expt[expt], linestyle="-", label=name_expt[expt], lw=1)

for expt in ['deglh', 'miroc', 'mpi_ice6g', 'mpi_glac1d']:
    axNGRIPsim.plot(time_sim[expt], util.rmean(ngrip[expt], 3) - util.rmean(ngrip[expt].values, 3)[0], 
                 color=color_expt[expt], linestyle="-", label=name_expt[expt], lw=1)
for expt in ['iloveclim_ice6g', 'iloveclim_glac1d']:
    axNGRIPsim.plot(ngrip[expt].time, util.rmean(ngrip[expt], 3) - util.rmean(ngrip[expt].values, 3)[0], 
                 color=color_expt[expt], linestyle="-", label=name_expt[expt], lw=1)

    
## NGRIP Proxy

axNGRIPproxy.plot(-proxy_ngrip.age, proxy_ngrip.temp - proxy_ngrip.temp[1066], color='black', label="Martin et al. (2023)")


# Parameters

axNotes.xaxis.set(ticks_position='top', label_position='top')
for loc in ['right', 'left', 'bottom']: axNotes.spines[loc].set_visible(False)
axNotes.set_xlabel("Years")
axNotes.set_xlim([-21500,-13000])
axNotes.yaxis.set_visible(False)

axMW.tick_params(axis='x', colors='None', which='both')
axMW.set_xlim([-21500,-13000])
axMW.set_ylabel("Total meltwater flux ($Sv$)", size='medium')
for loc in ['right', 'top', 'bottom']: axMW.spines[loc].set_visible(False)
axMW.legend(loc='upper left', fontsize='x-small', frameon=False, ncols=2)

axAMOChcm3.tick_params(axis='x', colors='None', which='both')
axAMOChcm3.yaxis.set(ticks_position='right', label_position='right')
axAMOChcm3.set_ylabel("AMOC index ($Sv$)", size='medium')
for loc in ['top','left','bottom']: axAMOChcm3.spines[loc].set_visible(False)
# axAMOChcm3.axhline(0, color='BLACK', linestyle='--', linewidth=0.5)
axAMOChcm3.legend(loc='lower left', fontsize='x-small', frameon=False)

axAMOCsim.tick_params(axis='x', colors='None', which='both')
axAMOCsim.set_ylabel("AMOC index ($Sv$)", size='medium')
for loc in ['top','right','bottom']: axAMOCsim.spines[loc].set_visible(False)
# axAMOCsim.axhline(0, color='BLACK', linestyle='--', linewidth=0.5)
axAMOCsim.legend(loc='upper left', fontsize='x-small', frameon=False, ncols=2,
                 bbox_to_anchor=(0.0, 0.1))

axAMOCpop.tick_params(axis='x', colors='None', which='both')
axAMOCpop.yaxis.set(ticks_position='right', label_position='right')
axAMOCpop.set_ylabel("AMOC index ($\%$ of PI)", size='medium')
# axAMOCpop.set_ylim([0.105,0.055])
for loc in ['top','left','bottom']: axAMOCpop.spines[loc].set_visible(False)
axNGRIPsim.axhline(100, color='BLACK', linestyle='--', linewidth=0.5)
axAMOCpop.legend(loc='lower left', fontsize='x-small', frameon=False)

axNGRIPhcm3.tick_params(axis='x', colors='None', which='both')
axNGRIPhcm3.set_ylabel("NGRIP temperature ($°C$)", size='medium')
for loc in ['top','right','bottom']: axNGRIPhcm3.spines[loc].set_visible(False)
axNGRIPhcm3.set_ylim([-12,15])
axNGRIPhcm3.axhline(0, color='BLACK', linestyle='--', linewidth=0.5)
# axNGRIPhcm3.legend(loc='lower left', fontsize='x-small', edgecolor='white')

axNGRIPsim.tick_params(axis='x', colors='None', which='both')
axNGRIPsim.set_ylabel("NGRIP temperature ($°C$)", size='medium')
axNGRIPsim.yaxis.set(ticks_position='right', label_position='right')
for loc in ['top','left','bottom']: axNGRIPsim.spines[loc].set_visible(False)
axNGRIPsim.axhline(0, color='BLACK', linestyle='--', linewidth=0.5)
# axNGRIPsim.legend(loc='lower left', fontsize='x-small', edgecolor='white')

axNGRIPproxy.set_xlabel("Years")
axNGRIPproxy.set_ylabel("NGRIP temperature ($°C$)", size='medium')
for loc in ['top','right']: axNGRIPproxy.spines[loc].set_visible(False)
axNGRIPproxy.axhline(0, color='BLACK', linestyle='--', linewidth=0.5)
axNGRIPproxy.legend(loc='lower left', fontsize='x-small', edgecolor='white')


for ax in axNotes, axMW, axAMOChcm3, axAMOCsim, axAMOCpop, axNGRIPhcm3, axNGRIPsim, axNGRIPproxy:
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.grid(which = "both", color='lightgrey', linestyle='--', linewidth=0.2)
    
    
# Spans
for ax in axNotes, axMW, axAMOChcm3, axAMOCsim, axAMOCpop, axNGRIPhcm3, axNGRIPsim, axNGRIPproxy:
    ax.axvspan(xmin=-17_800, xmax=-14_700, color=color_spans['cold'], alpha=0.15, linewidth=0)
    ax.axvspan(xmin=-14_700, xmax=-13_000, color=color_spans['merid'], alpha=0.15, linewidth=0)

axNotes.text(-16_000, 0.7, 'HS1', ha='center', va='top', color=color_spans['cold'], style='italic', size='small') 
axNotes.text(-13_800, 0.7, 'BAW', ha='center', va='top', color=color_spans['merid'], style='italic', size='small') 

i = count(0)
axMW.annotate(alc[next(i)], xy=(0.97,0.92), xycoords='axes fraction', size=12, weight='bold')
axAMOChcm3.annotate(alc[next(i)], xy=(0.02,0.92), xycoords='axes fraction', size=12, weight='bold')
axAMOCsim.annotate(alc[next(i)], xy=(0.02,0.9), xycoords='axes fraction', size=12, weight='bold')
axAMOCpop.annotate(alc[next(i)], xy=(0.97,0.1), xycoords='axes fraction', size=12, weight='bold')
axNGRIPhcm3.annotate(alc[next(i)], xy=(0.02,0.88), xycoords='axes fraction', size=12, weight='bold')
axNGRIPsim.annotate(alc[next(i)], xy=(0.02,0.88), xycoords='axes fraction', size=12, weight='bold')
axNGRIPproxy.annotate(alc[next(i)], xy=(0.02,0.88), xycoords='axes fraction', size=12, weight='bold')


In [ ]:
fig.savefig(f"{output_folder}/comparison.png", bbox_extra_artists=(), bbox_inches='tight', format='png')

## Figure - Zones

In [ ]:
ds_lsm = xr.open_dataset(f"{data_folder}/lgm_inputs/temev.qrparm.omask.nc")


In [ ]:
lon_lsm, lat_lsm, depth, lsm_example = ds_lsm.longitude.values, ds_lsm.latitude.values, ds_lsm.depthdepth.values, ds_lsm.lsm.values
masked = np.copy(lsm_example)  # land mask True (1) on land
depthm = np.ma.masked_less(depth, 500.0)  # mask areas shallower than 500m
masked_500m = np.copy(depthm.mask) + masked  # create binary mask from depth data
lat, lon = spreading.LatAxis(lat_lsm[:]), spreading.LonAxis(lon_lsm[:])
umgrid = spreading.Grid(lat, lon)

cb = spreading.generate_collection_boxes()
sr = spreading.generate_spreading_regions(cb, umgrid, masked, masked_500m)

In [ ]:
projection_map = ccrs.PlateCarree(central_longitude=-30.0)
fig, (axMW, axGLOB, axNA) = plt.subplots(nrows=1, ncols=3, subplot_kw={'projection':projection_map}, figsize=(14,6), dpi=300)


for ax in axMW, axGLOB, axNA:
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, alpha=0.5, linestyle='-')
    ax.pcolormesh(ds_lsm.longitude, ds_lsm.latitude,xr.where(lsm_example==0, np.nan,lsm_example),
                  cmap=ListedColormap(['xkcd:pale brown']), transform=ccrs.PlateCarree(),
                  zorder=1)

# Drawing zones

## Meltwater
label_mark = {'elwg':False, 'arc':False, 'pac':False, 'gin':False, 'med':False, 'so':False} 
for region in sr:
    if region['name'] in ['US_East_Coast', 'Gulf_of_Mexico', 'LabradorSea_BaffinBay']:
        cmap = ListedColormap([color_fluxes['elwg']])
    elif region['name'] in ['Greenland_Arctic', 'N_American_Arctic', 'Eurasian_Arctic', 'Siberian_Arctic']:
        cmap = ListedColormap([color_fluxes['arc']])
    elif region['name'] in ['East_Pacific', 'Russia_Pacific']:
        cmap = ListedColormap([color_fluxes['pac']])
    elif region['name'] in ['Atlantic_GreenlandIceland', 'EastGreenland_Iceland', 'EastIceland', 'UK_Atlantic', 'Eurasian_GINSeas', 'South_Iceland']:
        cmap = ListedColormap([color_fluxes['gin']])
    elif region['name'] in ['Mediterranean']:
        cmap = ListedColormap([color_fluxes['med']])
    elif region['name'] in ['Patagonia_Atlantic', 'Patagonia_Pacific', 'NorthNewZealand_Pacific', 'SouthNewZealand_Pacific',
                            'Antarctica_RossSea', 'Antarctica_AmundsenSea', 'Antarctica_WeddellSea', 'Antarctica_RiiserLarsonSea', 'Antarctica_DavisSea']:
        cmap = ListedColormap([color_fluxes['so']])
    
    axMW.pcolormesh(region['region'].grid.lon_center, region['region'].grid.lat_center, np.where(region['region'].mask, 1, np.nan), 
                     cmap=cmap, transform=ccrs.PlateCarree(), alpha=0.7)

for zone in ['elwg', 'arc', 'gin', 'pac', 'med', 'so']:
    axMW.fill([],[],
        color=color_fluxes[zone], alpha=0.5, fill=True,
        label=label_fluxes[zone], zorder=0, linewidth=0)

axMW.legend(fontsize='xx-small', loc='lower left')


## Global
for zone in masks.keys():
    axGLOB.pcolormesh(masks[zone].longitude, masks[zone].latitude, masks[zone],
                  cmap=ListedColormap([color_zones[zone]]), transform=ccrs.PlateCarree(), 
                  zorder=0, alpha=0.7)

    axGLOB.fill([],[],
            color=color_zones[zone], alpha=0.5, fill=True,
            label=label_zones[zone], zorder=0, linewidth=0)
        
axGLOB.legend(fontsize='xx-small', loc='lower left')

## North Atlatic
for zone in masks_na_atm.keys():
    axNA.pcolormesh(masks_na_atm[zone].longitude, masks_na_atm[zone].latitude, masks_na_atm[zone],
                  cmap=ListedColormap([color_zones[zone]]), transform=ccrs.PlateCarree(), 
                  zorder=0, alpha=0.7)

    axNA.fill([],[],
            color=color_zones[zone], alpha=0.5, fill=True,
            label=label_zones[zone], zorder=0, linewidth=0)
        
axNA.legend(fontsize='xx-small', loc='lower right')
axNA.set_extent([270,380,20,90])
    
i = count(0)
for ax in axMW, axGLOB, axNA:
        ax.annotate(alc[next(i)], xy=(0.03,0.92), xycoords='axes fraction', size=12, weight='bold')


In [ ]:
fig.savefig(f"{output_folder}/zones.png", bbox_extra_artists=(), bbox_inches='tight', format='png')

## Figure - Correlation and autocorrelation

In [ ]:
amoc, amocn = {}, {}

for expt in database.keys():
    amoc[expt] = xr.open_dataset(
        f"{database[expt]}/time_series/{expt}.merid.annual.nc").Merid_Atlantic.sel(
        latitude=26.5, method='nearest').max('depth')
    amocn[expt] = amoc[expt] - np.mean(amoc[expt])

amocn['xqctc'] = amocn['xqctc'].isel(t=slice(0,8500))

In [ ]:
fluxes, lsm = {}, {}
time = {}

In [ ]:
# ICE6G

ds_discharge = xr.open_dataset(f"{data_folder}/mw_inputs/xpfj.wfix.ice6g_ts.shift.nc",decode_times=False)
ds_discharge.attrs['waterfix'] = 'GLAC-1D'
ds_lsm = xr.open_dataset(f"{data_folder}/lgm_inputs/ice6g.omask.nc")
ds_wfix = xr.open_dataset(f"{data_folder}/lgm_inputs/teadv3.qrparm.waterfix.nc")

fluxes['ICE6G'] = plotting.create_discharge_ts(plotting.remove_waterfix(
    saving.ancil_to_discharge(ds_discharge), ds_wfix),
                                    ds_lsm, details='low', rmean=1)

lsm['ICE6G'] = ds_lsm.lsm

time['ICE6G'] = ds_discharge.t.values - 100000

In [ ]:
# GLAC-1D

ds_discharge = xr.open_dataset(f"{data_folder}/mw_inputs/xoup.wfix.glac_ts.nc",decode_times=False)
ds_discharge.attrs['waterfix'] = 'GLAC-1D'

ds_wfix = xr.open_dataset(f"{data_folder}/lgm_inputs/qrparm.waterfix.hadcm3.nc")
ds_lsm = xr.open_dataset(f"{data_folder}/lgm_inputs/temev.qrparm.omask.nc")

fluxes['GLAC-1D'] = plotting.create_discharge_ts(plotting.remove_waterfix(
    saving.ancil_to_discharge(ds_discharge), ds_wfix),
                                    ds_lsm, details='low', rmean=1)
lsm['GLAC-1D'] = ds_lsm.lsm

time['GLAC-1D'] = ds_discharge.t.values

In [ ]:
flux_dict, flux_dict_interp = {}, {}
for mw in ['GLAC-1D','ICE6G']:
    flux_dict[mw], flux_dict_interp[mw] = {}, {}
    for key in fluxes[mw].keys():
        flux_dict[mw][key] = fluxes[mw][key].value
        flux_dict_interp[mw][key] = np.interp(amoc['xpfjb'].t.dt.year.values-100_000, time[mw], flux_dict[mw][key]) - np.mean(flux_dict[mw][key])


In [ ]:
year_lags, cross_correlation, auto_correlation, norm_factor = {}, {}, {}, {}
    
for expt in ['xpfjb', 'xpfjc', 'xpfjd', 'xpfje', 'xqctc']:
    year_lags[expt] = scipy.signal.correlation_lags(len(amocn[expt]), len(amocn[expt]))
    norm_factor[expt] = len(amocn[expt]) * np.std(amocn[expt].values) ** 2
    auto_correlation[expt] = scipy.signal.correlate(amocn[expt], amocn[expt])/norm_factor[expt]

for expt in ['xpfjc', 'xpfjd', 'xqctc']:
    mw='GLAC-1D'
    cross_correlation[expt] = scipy.signal.correlate(-flux_dict_interp[mw]['tot'], amocn[expt])/norm_factor[expt]

for expt in ['xpfjb', 'xpfje']:
    mw='ICE6G'
    cross_correlation[expt] = scipy.signal.correlate(-flux_dict_interp[mw]['tot'], amocn[expt])/norm_factor[expt]

for expt in ['ICE6G', 'GLAC-1D']:
    year_lags[expt] = scipy.signal.correlation_lags(len(flux_dict_interp[expt]['tot']), len(flux_dict_interp[expt]['tot']))
    norm_factor[expt] = len(flux_dict_interp[expt]['tot']) * np.std(flux_dict_interp[expt]['tot']) * np.std(flux_dict_interp[expt]['tot'])
    auto_correlation[expt] = scipy.signal.correlate(flux_dict_interp[expt]['tot'], flux_dict_interp[expt]['tot'])
    auto_correlation[expt] = auto_correlation[expt]/np.max(auto_correlation[expt])


In [ ]:
fig = plt.figure(figsize=(12, 8), dpi=200)

axCC, axAutoCC = {}, {}

gs = gridspec.GridSpec(nrows=2, ncols=2, hspace=0, wspace=0.1)
axCC['GLAC-1D'] = fig.add_subplot(gs[0,0],facecolor='None')
axCC['ICE6G'] = fig.add_subplot(gs[1,0], sharex=axCC['GLAC-1D'], sharey=axCC['GLAC-1D'], facecolor='None')
axAutoCC['GLAC-1D'] = fig.add_subplot(gs[0,1],facecolor='None')
axAutoCC['ICE6G'] = fig.add_subplot(gs[1,1], sharex=axAutoCC['GLAC-1D'], sharey=axAutoCC['GLAC-1D'], facecolor='None')


for expt in ['xpfjc', 'xpfjd', 'xqctc']:
    axCC['GLAC-1D'].plot(year_lags[expt], cross_correlation[expt], color=color_expt[expt], label=name_expt[expt])
    axAutoCC['GLAC-1D'].plot(year_lags[expt], auto_correlation[expt], color=color_expt[expt], label=name_expt[expt])
axAutoCC['GLAC-1D'].plot(year_lags['GLAC-1D'], auto_correlation['GLAC-1D'], color='black', label='GLAC-1D discharge')

for expt in ['xpfjb', 'xpfje']:
    axCC['ICE6G'].plot(year_lags[expt], cross_correlation[expt], color=color_expt[expt], label=name_expt[expt])
    axAutoCC['ICE6G'].plot(year_lags[expt], auto_correlation[expt], color=color_expt[expt], label=name_expt[expt])
axAutoCC['ICE6G'].plot(year_lags['ICE6G'], auto_correlation['ICE6G'], color='black', label='ICE6G discharge')


# Parameters

axCC['GLAC-1D'].xaxis.set(ticks_position='top', label_position='top')
axCC['GLAC-1D'].set_xlabel("Lag years")
axCC['GLAC-1D'].set_ylabel("Correlation index (GLAC-1D)\nAMOC index vs discharge")
axCC['GLAC-1D'].legend(loc='upper left', fontsize='small', edgecolor='white')

axCC['ICE6G'].set_xlabel("Lag years")
axCC['ICE6G'].set_ylabel("Correlation index (ICE6G)\nAMOC index vs discharge")
axCC['ICE6G'].legend(loc='upper left', fontsize='small', edgecolor='white')

axAutoCC['GLAC-1D'].xaxis.set(ticks_position='top', label_position='top')
axAutoCC['GLAC-1D'].set_xlabel("Lag years")
axAutoCC['GLAC-1D'].set_ylabel("Auto-correlation index (GLAC-1D)\nAMOC index & discharge")
axAutoCC['GLAC-1D'].legend(loc='upper left', fontsize='small', edgecolor='white')
axAutoCC['GLAC-1D'].yaxis.set(ticks_position='right', label_position='right')

axAutoCC['ICE6G'].set_xlabel("Lag years")
axAutoCC['ICE6G'].set_ylabel("Auto-correlation index (ICE6G)\nAMOC index & discharge")
axAutoCC['ICE6G'].yaxis.set(ticks_position='right', label_position='right')
axAutoCC['ICE6G'].legend(loc='upper left', fontsize='small', edgecolor='white')

i = count(0)
for mw in ['GLAC-1D', 'ICE6G']:
    
    axCC[mw].set_xlim([-2000,2000])

    axCC[mw].xaxis.set_minor_locator(AutoMinorLocator())
    axCC[mw].yaxis.set_minor_locator(AutoMinorLocator())
    axCC[mw].grid(which = "both", color='lightgrey', linestyle='--', linewidth=0.2)
    axCC[mw].annotate(alc[next(i)], xy=(0.95,0.9), xycoords='axes fraction', size=12, weight='bold')
    axCC[mw].xaxis.set_minor_locator(AutoMinorLocator())
    axCC[mw].axvline(0, color='black', linestyle='--', linewidth=0.5)

    axCC[mw].annotate('AMOC leading', xy=(0.9, 0.97), xycoords='axes fraction',
                        size=7, va='top', ha='right', style='italic')

for mw in ['GLAC-1D', 'ICE6G']:

    axAutoCC[mw].set_xlim([-5000,5000])
    
    axAutoCC[mw].xaxis.set_minor_locator(AutoMinorLocator())
    axAutoCC[mw].yaxis.set_minor_locator(AutoMinorLocator())
    axAutoCC[mw].grid(which = "both", color='lightgrey', linestyle='--', linewidth=0.2)
    axAutoCC[mw].annotate(alc[next(i)], xy=(0.95,0.95), xycoords='axes fraction', size=12, weight='bold')
    axAutoCC[mw].xaxis.set_minor_locator(AutoMinorLocator())

    axAutoCC[mw].axvline(0, color='black', linestyle='--', linewidth=0.5)
    

In [ ]:
fig.savefig(f"{output_folder}/correlation.png", bbox_extra_artists=(), bbox_inches='tight', format='png')

## Figure - Ice-sheet reconstructions and atmospheric circulation (expanded)

In [ ]:
lsm = {}

ds_lsm = xr.open_dataset(f"{data_folder}/lgm_inputs/ice6g.omask.nc")
lsm['ICE6G'] = ds_lsm.lsm

ds_lsm = xr.open_dataset(f"{data_folder}/lgm_inputs/temev.qrparm.omask.nc")
lsm['GLAC-1D'] = ds_lsm.lsm


In [ ]:
means = {}

# SST
means['sst'] = {}
means['sst']['tecac'] = xr.open_dataset(f"{database['tecac']}/control_summaries/{'tecac'}.sst.annual.summary.nc").temp_1
means['sst']['xoupa'] = xr.open_dataset(f"{database['xoupa']}/control_summaries/{'xoupa'}.sst.annual.summary.nc").temp_mm_dpth

# SAT
means['sat'] = {}
means['sat']['tecac'] = xr.open_dataset(f"{database['tecac']}/control_summaries/{'tecac'}.sat.annual.summary.nc").sat -273.15
means['sat']['xoupa'] = xr.open_dataset(f"{database['xoupa']}/control_summaries/{'xoupa'}.sat.annual.summary.nc").sat -273.15


# Mean sea level pressure
means['mslp'] = {}
means['mslp']['tecac'] = util.extend_lon(
    xr.open_dataset(f"{database['tecac']}/control_summaries/{'tecac'}.mslp.annual.summary.nc").p, 1, 'longitude')/100
means['mslp']['xoupa'] = util.extend_lon(
    xr.open_dataset(f"{database['xoupa']}/control_summaries/{'xoupa'}.mslp.annual.summary.nc").p_mm_msl, 1, 'longitude')/100

# Geopotential height
means['z500'] = {}
means['z500']['tecac'] = util.extend_lon(
    xr.open_dataset(f"{database['tecac']}/control_summaries/{'tecac'}.z500.annual.summary.nc").ht, 1, 'longitude')/1000
means['z500']['xoupa'] = util.extend_lon(
    xr.open_dataset(f"{database['xoupa']}/control_summaries/{'xoupa'}.z500.annual.summary.nc").ht_mm_p, 1, 'longitude')/1000

# Wind stress at 850/200 hPa
means['u850'], means[f'v850'], means[f'ws850'] = {}, {}, {}
means['u200'], means[f'v200'], means[f'ws200'] = {}, {}, {}
for expt in ['tecac', 'xoupa']:
    
    for z in ['200', '850']:
        means[f'u{z}'][expt] = xr.open_dataset(f"{database[expt]}/control_summaries/{expt}.u{z}.annual.summary.nc").u
        means[f'v{z}'][expt] = xr.open_dataset(f"{database[expt]}/control_summaries/{expt}.v{z}.annual.summary.nc").v
        means[f'ws{z}'][expt] = xr.open_dataset(f"{database[expt]}/control_summaries/{expt}.ws{z}.annual.summary.nc").ws


In [ ]:
cmaps = {'sst': scm.cm.vik, 'sst_diff':scm.cm.vik,
         'sat': scm.cm.vik, 'sat_diff':scm.cm.vik,
         'mslp':'Greens', 'mslp_diff':scm.cm.bam,
         'z500':'Greens', 'z500_diff':scm.cm.bam, 
         'ws850':ListedColormap(scm.cm.broc(np.linspace(0.5, 1, 128))), 'ws850_diff':scm.cm.broc,
         'ws200':ListedColormap(scm.cm.broc(np.linspace(0.5, 1, 128))), 'ws200_diff':scm.cm.broc}

norms = {'sst': TwoSlopeNorm(vmin=-2, vcenter=0, vmax=30), 'sst_diff':TwoSlopeNorm(vmin=-10, vcenter=0, vmax=10),
         'sat': TwoSlopeNorm(vmin=-50, vcenter=0, vmax=30), 'sat_diff':TwoSlopeNorm(vmin=-15, vcenter=0, vmax=15),
         'mslp': Normalize(vmin=1010, vmax=1040), 'mslp_diff':TwoSlopeNorm(vmin=-10, vcenter=0, vmax=10),
         'z500': Normalize(vmin=5.1, vmax=5.6), 'z500_diff':TwoSlopeNorm(vmin=-.1, vcenter=0, vmax=.1), 
         'ws850': Normalize(vmin=0, vmax=20), 'ws850_diff': TwoSlopeNorm(vmin=-10, vcenter=0, vmax=10),
         'ws200': Normalize(vmin=0, vmax=50), 'ws200_diff': TwoSlopeNorm(vmin=-10, vcenter=0, vmax=10)}


In [ ]:
projection_map = ccrs.NearsidePerspective(central_longitude=-30, central_latitude=70, satellite_height=8000000)

axICE6G, axGLAC1D, axDIFF = {}, {}, {}

fig = plt.figure(figsize=(15, 22), dpi=300)
axHIST = {}

gs = fig.add_gridspec(
    nrows=6, ncols=1,
    height_ratios=[1, 1, 1, 1, 1, 1],
    wspace=0.2, hspace=0.1
)

for row, var in enumerate(['sst', 'sat', 'mslp', 'z500', 'ws850', 'ws200']):
    subgs = gs[row].subgridspec(1, 3, wspace=0.05)
    axICE6G[var] = fig.add_subplot(subgs[0, 0], projection=projection_map)
    axGLAC1D[var] = fig.add_subplot(subgs[0, 1], projection=projection_map)
    axDIFF[var] = fig.add_subplot(subgs[0, 2], projection=projection_map)

# -----

# Plot variables
labels_cbar = {'sst':"Sea surface temperature\n($^\circ C$)", 
               'sat':"Surface atmosphere temperature\n($^\circ C$)",
               'mslp':"Mean sea level pressure\n($hPa$)",
               'z500': "Geopotential height at 500 hPa\n($km$)",
               'ws850':"Wind stress at 850 hPa\n($m.s^{-1}$)",
               'ws200':"Wind stress at 200 hPa\n($m.s^{-1}$)"}

for var in ['sst', 'sat', 'mslp', 'z500', 'ws850', 'ws200']:        
    axICE6G[var].pcolormesh(means[var]['tecac'].longitude, means[var]['tecac'].latitude, 
                            means[var]['tecac'], zorder=1,
                            cmap=cmaps[var], norm=norms[var], transform=ccrs.PlateCarree())

    if var == 'sst':
        cbar = fig.colorbar(mappable=matplotlib.cm.ScalarMappable(cmap=cmaps[var], norm=norms[var]),
                            ax=axICE6G[var], ticks = [-2,-1,0,10,20,30], 
                            orientation='vertical', shrink=0.8, pad=0.05)
    elif var == 'sat':
        cbar = fig.colorbar(mappable=matplotlib.cm.ScalarMappable(cmap=cmaps[var], norm=norms[var]),
                            ax=axICE6G[var], ticks = [-40,-20,0,10,20,30], 
                            orientation='vertical', shrink=0.8, pad=0.05)
    else:
        cbar = fig.colorbar(mappable=matplotlib.cm.ScalarMappable(cmap=cmaps[var], norm=norms[var]),
                                ax=axICE6G[var],
                                orientation='vertical', shrink=0.8, pad=0.05)
    
    axGLAC1D[var].pcolormesh(means[var]['xoupa'].longitude, means[var]['xoupa'].latitude, 
                             means[var]['xoupa'], zorder=1,
                             cmap=cmaps[var], norm=norms[var], transform=ccrs.PlateCarree())

    if var == 'sst':
        cbar = fig.colorbar(mappable=matplotlib.cm.ScalarMappable(cmap=cmaps[var], norm=norms[var]),
                            ax=axGLAC1D[var], ticks = [-2,-1,0,10,20,30], 
                            orientation='vertical', shrink=0.8, pad=0.05)
    elif var == 'sat':
        cbar = fig.colorbar(mappable=matplotlib.cm.ScalarMappable(cmap=cmaps[var], norm=norms[var]),
                            ax=axGLAC1D[var], ticks = [-40,-20,0,10,20,30], 
                            orientation='vertical', shrink=0.8, pad=0.05)
    else:
        cbar = fig.colorbar(mappable=matplotlib.cm.ScalarMappable(cmap=cmaps[var], norm=norms[var]),
                            ax=axGLAC1D[var],
                            orientation='vertical', shrink=0.8, pad=0.05)
    

    axDIFF[var].pcolormesh(means[var]['xoupa'].longitude, means[var]['xoupa'].latitude, 
                            means[var]['tecac'] - means[var]['xoupa'], zorder=1,
                            cmap=cmaps[var+'_diff'], norm=norms[var+'_diff'], transform=ccrs.PlateCarree())
    cbar = fig.colorbar(mappable=matplotlib.cm.ScalarMappable(cmap=cmaps[var+'_diff'], norm=norms[var+'_diff']),
                           ax=axDIFF[var],
                           orientation='vertical', shrink=0.8, pad=0.05)
    cbar.set_label(labels_cbar[var], size='large')


# Plot land-sea masks    
for ax in axICE6G.values():
    ax.pcolormesh(lsm['ICE6G'].longitude, lsm['ICE6G'].latitude,
                  xr.where(lsm['ICE6G']==0, np.nan, lsm['ICE6G']),
                  zorder=0,
                  transform=ccrs.PlateCarree(), cmap=ListedColormap(['xkcd:pale brown']))

for ax in axGLAC1D.values():
    ax.pcolormesh(lsm['GLAC-1D'].longitude, lsm['GLAC-1D'].latitude,
                  xr.where(lsm['GLAC-1D']==0, np.nan, lsm['GLAC-1D']),
                  zorder=0,
                  transform=ccrs.PlateCarree(), cmap=ListedColormap(['xkcd:pale brown']))

for ax in axDIFF.values():
    ax.pcolormesh(lsm['GLAC-1D'].longitude, lsm['GLAC-1D'].latitude,
                  xr.where(lsm['GLAC-1D']+lsm['ICE6G']==0, np.nan, lsm['GLAC-1D']+lsm['ICE6G']),
                  zorder=0, 
                  transform=ccrs.PlateCarree(), cmap=ListedColormap(['xkcd:pale brown']))

for ax in [axICE6G, axGLAC1D, axDIFF]: 
    ax['sat'].coastlines(lw=0.5)
    ax['mslp'].coastlines(lw=0.5)
    ax['z500'].coastlines(lw=0.5)
    ax['ws850'].coastlines(lw=0.5)
    ax['ws200'].coastlines(lw=0.5)

# -----

# Experiment Box
def plot_experiment_box(name, ax, color, size='x-large'):
    ax_box = ax.get_position()
    
    cax = fig.add_axes([ax_box.x0, ax_box.y1+0.02,
                    ax_box.width+0.02, 
                    ax_box.height/4])
    
    cax.get_xaxis().set_visible(False)
    cax.get_yaxis().set_visible(False)
    for loc in ['left', 'right', 'bottom', 'top']: cax.spines[loc].set_visible(False)
    cax.set_facecolor(color)

    at = AnchoredText(name, loc=10, frameon=False,
                        prop=dict(backgroundcolor=color,
                                  size=size, color='white'))
    
    cax.add_artist(at)
    
plot_experiment_box(name_expt['tecac'], axICE6G['sst'], color_expt['tecac'])
plot_experiment_box(name_expt['xoupa'], axGLAC1D['sst'], color_expt['xoupa'])
plot_experiment_box(name_expt['tecac'] + ' - ' + name_expt['xoupa'], axDIFF['sst'], 'darkslategrey', size='large')


i = count(0)
for var in ['sst', 'sat', 'mslp', 'z500', 'ws850', 'ws200']:
    for ax in axICE6G, axGLAC1D, axDIFF:
        txt = ax[var].annotate(alc[next(i)], xy=(-0.02,0.87), xycoords='axes fraction', size=12, weight='bold')


In [ ]:
fig.savefig(f"{output_folder}/atmospheric_expanded.png", bbox_inches='tight', format='png')

## Figure - Atmospheric variables distribution

In [ ]:
bands = {}

bands['arc'] = [(0,360), (80,90)]
bands['high_na'] = [(335,380), (55,80)]
bands['low_na'] = [(335,380), (35,50)]

label_band = {'arc':'Arctic (>80° N)', 'high_na':"North Atlantic (55° N-80° N)", 'low_na':"North Atlantic (35° N-50° N)"}

In [ ]:
def compute_jet_index(ds_u850, lon_min, lon_max, lat_min, lat_max):

    ds_sliced = ds_u850.sel(longitude=slice(lon_min, lon_max)).sel(latitude=slice(lat_min, lat_max))

    # Vectorised computation over years (no Python loop)
    # ds_test dims: (year, latitude, longitude)

    # latitude of max wind per (year, longitude)
    lat_at_max = ds_sliced.idxmax(dim='latitude')  # DataArray (year, longitude) with latitude coord values

    # jet latitude: mean latitude of maxima across longitudes (per year)
    jet_lats = lat_at_max.mean(dim='longitude').values

    # jet strength: mean wind (averaged over latitude then longitude) per year
    mean_wind_per_lon = ds_sliced.mean(dim='latitude')         # (year, longitude)
    jet_strengths = mean_wind_per_lon.mean(dim='longitude').values

    # jet angle: slope of linear fit longitude vs lat_at_max for each year (vectorised)
    x = ds_sliced.longitude.values.astype(float)               # (nlon,)
    x_mean = x.mean()
    y = lat_at_max.values.astype(float)                      # (nyear, nlon)
    y_mean = y.mean(axis=1)                                  # (nyear,)

    denom = np.sum((x - x_mean) ** 2)
    slopes = np.sum((x - x_mean) * (y - y_mean[:, None]), axis=1) / denom
    jet_angles = np.degrees(np.arctan(slopes)).tolist()

    return jet_lats, jet_strengths, jet_angles

In [ ]:
data = {}

for expt in ['xoupa', 'tecac', 'xpfjb', 'xpfjc', 'xpfjd', 'xpfje']:

    data[expt] = {}
            
    # Winter surface winds
    print(f"__ Importing winter surface winds stress for {expt}")
    
    ds_ws = xr.open_dataset(
        f"{database[expt]}/time_series/na_variables/{expt}.ws10_zone.zone_na.winter.annual.nc").ws10
    data[expt]['ws10'] = {}
    for zone in ['nor', 'icd', 'irm', 'ls']:
        data[expt]['ws10'][zone] = ds_ws.sel(zone=zone)
    
    # MSLP
    print(f"__ Importing Mean sea level pressure for {expt}")
    ds = util.extend_lon(xr.open_dataset(
        f"{database[expt]}/time_series/{expt}.mslp.annual.nc").p_mm_msl, 10, 'longitude')
    data[expt]['mslp'] = {}
    for zone in bands.keys():
        data[expt]['mslp'][zone] = ds.sel(
            longitude=slice(*bands[zone][0])).sel(latitude=slice(*bands[zone][1])).mean(['longitude', 'latitude'])/100

    # EDJ index
    print(f"__ Importing winter jet indexes for {expt}")
    data[expt]['jet_lat'], data[expt]['jet_max'], data[expt]['jet_angles'] = {}, {}, {}
    lon_min, lon_max = 280, 360
    lat_min, lat_max = 25, 65

    ds_u200 = xr.open_dataset(
        f"{database[expt]}/time_series/{expt}.u200.winter.nc").u_mm_p
    ds_u850 = xr.open_dataset(
        f"{database[expt]}/time_series/{expt}.u850.winter.nc").u_mm_p

    jet_indexes = compute_jet_index(ds_u200, lon_min, lon_max, lat_min, lat_max)
    data[expt]['jet_lat']['850'] = jet_indexes[0]
    data[expt]['jet_max']['850'] = jet_indexes[1]
    data[expt]['jet_angles']['850'] = jet_indexes[2]


In [ ]:
kernel = {}

for expt in ['xoupa', 'tecac', 'xpfjb', 'xpfjc', 'xpfjd', 'xpfje']:
    print(f"__ Processing {expt}")
    kernel[expt] = {}
        
    for var in ['jet_lat', 'jet_max', 'jet_angles']:
        kernel[expt][var] = {}
        for zone in ['850']:
            kernel[expt][var][zone] = stats.gaussian_kde(data[expt][var][zone])

    var ='mslp'
    kernel[expt][var] = {}
    for zone in ['arc', 'high_na', 'low_na']:
        kernel[expt][var][zone] = stats.gaussian_kde(data[expt][var][zone])

    var ='ws10'
    kernel[expt][var] = {}
    for zone in ['nor', 'icd', 'irm', 'ls']:
        kernel[expt][var][zone] = stats.gaussian_kde(data[expt][var][zone])

In [ ]:
lw = {'xoupa':2, 'tecac':2, 'xpfjb':1, 'xpfjc':1, 'xpfjd':1, 'xpfje':1}
zo = {'xoupa':2, 'tecac':2, 'xpfjb':1, 'xpfjc':1, 'xpfjd':1, 'xpfje':1}

In [ ]:
fig = plt.figure(figsize=(10, 10), dpi=300)
axHIST = {}

gs = fig.add_gridspec(
    nrows=3, ncols=1,
    height_ratios=[1, 1, 1],
    wspace=0.2, hspace=0.1
)
nbins = 50
alpha_hist = 0.05

# Jet lat & max Grid
axHIST['jet'] = {}
subgs = gs[0].subgridspec(1, 3, wspace=0.05)
axHIST['jet']['jet_lat'] = fig.add_subplot(subgs[0])
axHIST['jet']['jet_max'] = fig.add_subplot(subgs[1], sharey=axHIST['jet']['jet_lat'])
axHIST['jet']['jet_angles'] = fig.add_subplot(subgs[2], sharey=axHIST['jet']['jet_lat'])

        
# Mean sea level pressure
axHIST['mslp'] = {}
subgs = gs[1].subgridspec(1, 3, wspace=0.05)
axHIST['mslp']['arc'] = fig.add_subplot(subgs[0])
axHIST['mslp']['high_na'] = fig.add_subplot(subgs[1], sharey=axHIST['mslp']['arc'])
axHIST['mslp']['low_na'] = fig.add_subplot(subgs[2], sharey=axHIST['mslp']['arc'])

axHIST['ws10'] = {}
subgs = gs[2].subgridspec(1, 3, wspace=0.05)
axHIST['ws10']['nor'] = fig.add_subplot(subgs[0], facecolor='None')
axHIST['ws10']['icd'] = fig.add_subplot(subgs[1], facecolor='None')
axHIST['ws10']['irm'] = fig.add_subplot(subgs[2], facecolor='None')

# -----

# Plots Jet indexes 
for var in ['jet_lat', 'jet_max', 'jet_angles']:
    for expt in ['tecac', 'xoupa', 'xpfjb', 'xpfjc','xpfjd', 'xpfje']:
        axHIST['jet'][var].hist(data[expt][var]['850'], bins=nbins,
                color=color_expt[expt], alpha=alpha_hist, density=True)
        x = np.linspace(axHIST['jet'][var].get_xlim()[0], axHIST['jet'][var].get_xlim()[1], nbins)
        axHIST['jet'][var].plot(x,kernel[expt][var]['850'](x), color=color_expt[expt], lw=lw[expt], zorder=zo[expt])

axHIST['jet']['jet_lat'].text(.02, .98, f"EDJ latitude index at 850hPa (°N)", 
                        ha='left', va='top', fontsize='small',
                        transform=axHIST['jet']['jet_lat'].transAxes)
axHIST['jet']['jet_max'].text(.02, .98, f"EDJ strength index at 850hPa (m/s)", 
                            ha='left', va='top', fontsize='small',
                            transform=axHIST['jet']['jet_max'].transAxes) 
axHIST['jet']['jet_angles'].text(.02, .98, f"EDJ angle index at 850hPa (°)", 
                            ha='left', va='top', fontsize='small',
                            transform=axHIST['jet']['jet_angles'].transAxes)
axHIST['jet']['jet_lat'].yaxis.set_visible(False)  
axHIST['jet']['jet_max'].yaxis.set_visible(False)
axHIST['jet']['jet_angles'].yaxis.set_ticks_position('right')  
axHIST['jet']['jet_angles'].yaxis.set_label_position('right')  
axHIST['jet']['jet_angles'].set_ylabel("Probability distribution")


# Plots MSLP 
for expt in ['tecac', 'xoupa', 'xpfjb', 'xpfjc','xpfjd', 'xpfje']:
    for zone in bands.keys():
        axHIST['mslp'][zone].hist(data[expt]['mslp'][zone], bins=nbins,
                color=color_expt[expt], alpha=alpha_hist, density=True)
        axHIST['mslp'][zone].plot([],[], color=color_expt[expt])
        x = np.linspace(axHIST['mslp'][zone].get_xlim()[0], axHIST['mslp'][zone].get_xlim()[1], nbins)
        axHIST['mslp'][zone].plot(x,kernel[expt]['mslp'][zone](x), color=color_expt[expt], lw=lw[expt], zorder=zo[expt], label=name_expt[expt])

for band in axHIST['mslp'].keys():   
    axHIST['mslp'][band].text(.02, .98, label_band[band], 
                              ha='left', va='top', fontsize='small', 
                              transform=axHIST['mslp'][band].transAxes) 
axHIST['mslp']['arc'].set_ylabel("Mean sea level pressure\nhPa", fontsize='medium')
axHIST['mslp']['arc'].yaxis.set_visible(False)  
axHIST['mslp']['high_na'].yaxis.set_visible(False)  
axHIST['mslp']['low_na'].yaxis.set_ticks_position('right')
axHIST['mslp']['low_na'].yaxis.set_label_position('right')
axHIST['mslp']['low_na'].set_ylabel("Probability distribution")

    
# Plots wind stress
for expt in ['tecac', 'xoupa', 'xpfjb', 'xpfjc','xpfjd', 'xpfje']:
    for zone in ['nor', 'icd', 'irm']:
        axHIST['ws10'][zone].hist(data[expt]['ws10'][zone], bins=nbins,
             color=color_expt[expt], alpha=alpha_hist, density=True)
        x = np.linspace(axHIST['ws10'][zone].get_xlim()[0], axHIST['ws10'][zone].get_xlim()[1], nbins)
        axHIST['ws10'][zone].plot(x,kernel[expt]['ws10'][zone](x), color=color_expt[expt], lw=lw[expt], zorder=zo[expt], label=name_expt[expt])

for band in axHIST['ws10'].keys():   
    axHIST['ws10'][band].text(.02, .98, label_zones[band], 
                              ha='left', va='top', fontsize='small', 
                              transform=axHIST['ws10'][band].transAxes) 
axHIST['ws10']['nor'].set_ylabel("Mean sea level pressure\nhPa", fontsize='medium')
axHIST['ws10']['nor'].yaxis.set_visible(False)  
axHIST['ws10']['icd'].yaxis.set_visible(False)  
axHIST['ws10']['irm'].yaxis.set_ticks_position('right')
axHIST['ws10']['irm'].yaxis.set_label_position('right')
axHIST['ws10']['irm'].set_ylabel("Probability distribution")


# -----

# Spines
for loc in ['left', 'right', 'top']:
    axHIST['jet']['jet_lat'].spines[loc].set_visible(False)
    axHIST['jet']['jet_max'].spines[loc].set_visible(False)
    axHIST['mslp']['arc'].spines[loc].set_visible(False)
    axHIST['mslp']['high_na'].spines[loc].set_visible(False)
    axHIST['ws10']['nor'].spines[loc].set_visible(False)
    axHIST['ws10']['icd'].spines[loc].set_visible(False)

for loc in ['left', 'top']:
    axHIST['jet']['jet_angles'].spines[loc].set_visible(False)
    axHIST['mslp']['low_na'].spines[loc].set_visible(False)
    axHIST['ws10']['irm'].spines[loc].set_visible(False)

# Annotations

axHIST['mslp']['arc'].legend(loc='upper left', bbox_to_anchor=(0,0.9), frameon=False, fontsize='x-small')

i = count(0)
for var in ['jet_lat', 'jet_max', 'jet_angles']:
    txt = axHIST['jet'][var].annotate(alc[next(i)], xy=(0.9,0.8), xycoords='axes fraction', size=12, weight='bold')
for band in axHIST['mslp'].keys():   
    txt = axHIST['mslp'][band].annotate(alc[next(i)], xy=(0.9,0.8), xycoords='axes fraction', size=12, weight='bold')
for band in axHIST['ws10'].keys():   
    txt = axHIST['ws10'][band].annotate(alc[next(i)], xy=(0.9,0.8), xycoords='axes fraction', size=12, weight='bold')



In [ ]:
fig.savefig(f"{output_folder}/distribution.png", bbox_extra_artists=(), format='png')

## Figure - Impact of the Icelandic ice sheet

In [ ]:
database_icd = {'xptxd':f"{data_folder}/database/xptxd",
                'xptxe':f"{data_folder}/database/xptxe"}

In [ ]:
lsm = {}

ds_lsm = xr.open_dataset(f"{data_folder}/mw_protocol/data/ice6g.omask.nc")
lsm['ICE6G'] = ds_lsm.lsm

ds_lsm = xr.open_dataset(f"{data_folder}/lgm_inputs/temev.qrparm.omask.nc")
lsm['GLAC-1D'] = ds_lsm.lsm


In [ ]:
orog_ICE6G = xr.open_dataset(f"{ice_sheet_folder}/ICE6G_21_0k/qrparm.orog.nc").isel(t=0).isel(surface=0).ht
orog_ICE6G = xr.where(orog_ICE6G==0,np.nan,orog_ICE6G)
orog_ICE6G_ICD = xr.open_dataset(f"{ice_sheet_folder}/ICE6G_21_0k_noICD/qrparm.orog.nc").isel(t=0).isel(surface=0).ht
orog_ICE6G_ICD = xr.where(orog_ICE6G_ICD==0,np.nan,orog_ICE6G_ICD)

orog_anomaly = xr.where(orog_ICE6G - orog_ICE6G_ICD==0, np.nan, orog_ICE6G - orog_ICE6G_ICD)

In [ ]:
expt_icd = 'xptxd'
expt_no_icd = 'xptxe'
year_range = 100

In [ ]:
amoc = {}

for expt in [expt_icd, expt_no_icd]:
    amoc[expt]= xr.open_dataset(f"{database_icd[expt]}/time_series/{expt}.merid.annual.nc").Merid_Atlantic.sel(latitude=26.5, method='nearest').max('depth')

In [ ]:
mld_icd = xr.open_dataset(
    f"{database_icd[expt_icd]}/time_series/{expt_icd}.oceanmixedpf.monthly.nc").mixLyrDpth_mm_uo.isel(unspecified=0).isel(
    t=slice(-year_range*12,-1)).groupby("t.season").mean('t').sel(season='DJF')
mld_icd = xr.where(mld_icd==0,np.nan,mld_icd)

mld_no_icd = xr.open_dataset(
    f"{database_icd[expt_no_icd]}/time_series/{expt_no_icd}.oceanmixedpf.monthly.nc").mixLyrDpth_mm_uo.isel(unspecified=0).isel(
    t=slice(-year_range*12,-1)).groupby("t.season").mean('t').sel(season='DJF')
mld_no_icd = xr.where(mld_no_icd==0,np.nan,mld_no_icd)


In [ ]:
u_icd = xr.open_dataset(
    f"{database_icd[expt_icd]}/time_series/{expt_icd}.u10m.monthly.nc").u_mm_10m.isel(ht=0).groupby('t.year').mean('t').isel(year=slice(-year_range,-1)).mean('year')
v_icd = xr.open_dataset(
    f"{database_icd[expt_icd]}/time_series/{expt_icd}.v10m.monthly.nc").v_mm_10m.isel(ht=0).groupby('t.year').mean('t').isel(year=slice(-year_range,-1)).mean('year')
ws_icd = np.sqrt(u_icd**2 + v_icd**2)
ws_icd = xr.where(ws_icd==0,np.nan,ws_icd)

u_no_icd = xr.open_dataset(
    f"{database_icd[expt_no_icd]}/time_series/{expt_no_icd}.u10m.monthly.nc").u_mm_10m.isel(ht=0).groupby('t.year').mean('t').isel(year=slice(-year_range,-1)).mean('year')
v_no_icd = xr.open_dataset(
    f"{database_icd[expt_no_icd]}/time_series/{expt_no_icd}.v10m.monthly.nc").v_mm_10m.isel(ht=0).groupby('t.year').mean('t').isel(year=slice(-year_range,-1)).mean('year')
ws_no_icd = np.sqrt(u_no_icd**2 + v_no_icd**2)
ws_no_icd = xr.where(ws_icd==0,np.nan,ws_no_icd)


In [ ]:
cmaps = {'mld':'Greens', 'mld_diff':scm.cm.cork,
         'ice': 'Blues_r', 'ice_diff':scm.cm.vik.reversed(),
         'mslp':'Greens', 'mslp_diff':scm.cm.bam,
         'ws':matplotlib.colors.ListedColormap(scm.cm.broc(np.linspace(0.5, 1, 128))), 'ws_diff':scm.cm.broc}

norms = {'mld': Normalize(vmin=0, vmax=600), 'mld_diff':TwoSlopeNorm(vmin=-400, vcenter=0, vmax=400),
         'sst': TwoSlopeNorm(vmin=-2, vcenter=0, vmax=30), 'sst_diff':TwoSlopeNorm(vmin=-10, vcenter=0, vmax=10),
         'ice': Normalize(vmin=0, vmax=4000), 'ice_diff':TwoSlopeNorm(vmin=-3000, vcenter=0, vmax=3000),
         'mslp': Normalize(vmin=1010, vmax=1040), 'mslp_diff':TwoSlopeNorm(vmin=-10, vcenter=0, vmax=10),
         'ws': Normalize(vmin=0, vmax=10), 'ws_diff': TwoSlopeNorm(vmin=-5, vcenter=0, vmax=5)}

In [ ]:
projection_map = ccrs.NearsidePerspective(central_longitude=-30, central_latitude=70, satellite_height=8000000)

fig = plt.figure(figsize=(15,4), dpi=300)

gs = gridspec.GridSpec(nrows=1, ncols=4, wspace=0.1, hspace=0.1,width_ratios=[5,4,4,4])
axAMOC = fig.add_subplot(gs[0])
axOrog = fig.add_subplot(gs[1], projection = projection_map)
axMLD = fig.add_subplot(gs[2], projection = projection_map)
axWS = fig.add_subplot(gs[3], projection = projection_map)

axAMOC.plot(amoc['xptxd'].t.dt.year, util.rmean(amoc['xptxd'],30), color=color_expt['tecac'], label=name_expt['tecac'])
axAMOC.plot(amoc['xptxd'].t.dt.year, amoc['xptxd'], color=color_expt['tecac'], alpha=0.2)
axAMOC.plot(amoc['xptxe'].t.dt.year, util.rmean(amoc['xptxe'],30), color='xkcd:forrest green', label='ICE6G_noMw_noIcd')
axAMOC.plot(amoc['xptxe'].t.dt.year, amoc['xptxe'], color='xkcd:forrest green', alpha=0.2)

axAMOC.set_xlabel('Simulation years')
axAMOC.set_ylabel('AMOC index (Sv)')
for loc in ['top','right']: axAMOC.spines[loc].set_visible(False)
axAMOC.legend(loc='lower left', fontsize='xx-small', edgecolor='white')

axOrog.pcolormesh(orog_ICE6G.longitude, orog_ICE6G.latitude, 
                  orog_anomaly, zorder=1,
                  cmap=cmaps['ice'+'_diff'], norm=norms['ice'+'_diff'], transform=ccrs.PlateCarree())
cbar = fig.colorbar(mappable=matplotlib.cm.ScalarMappable(cmap=cmaps['ice'+'_diff'], norm=norms['ice'+'_diff']),
                    ax=axOrog,
                    orientation='vertical', shrink=0.5, pad=0.05)


axMLD.pcolormesh(mld_icd.longitude, mld_icd.latitude, 
                 mld_icd - mld_no_icd, zorder=1,
                 cmap=cmaps['mld'+'_diff'], norm=norms['mld'+'_diff'], transform=ccrs.PlateCarree())
cbar = fig.colorbar(mappable=matplotlib.cm.ScalarMappable(cmap=cmaps['mld'+'_diff'], norm=norms['mld'+'_diff']),
                    ax=axMLD,
                    orientation='vertical', shrink=0.5, pad=0.05)

axWS.pcolormesh(ws_icd.longitude_1, ws_icd.latitude_1, 
                 ws_icd - ws_no_icd, zorder=1,
                 cmap=cmaps['ws'+'_diff'], norm=norms['ws'+'_diff'], transform=ccrs.PlateCarree())
cbar = fig.colorbar(mappable=matplotlib.cm.ScalarMappable(cmap=cmaps['ws'+'_diff'], norm=norms['ws'+'_diff']),
                    ax=axWS,
                    orientation='vertical', shrink=0.5, pad=0.05)

axOrog.pcolormesh(lsm['ICE6G'].longitude, lsm['ICE6G'].latitude,
              xr.where(lsm['ICE6G']==0, np.nan, lsm['ICE6G']),
              zorder=0,
              transform=ccrs.PlateCarree(), cmap=ListedColormap(['xkcd:pale brown']))

axMLD.pcolormesh(lsm['ICE6G'].longitude, lsm['ICE6G'].latitude,
              xr.where(lsm['ICE6G']==0, np.nan, lsm['ICE6G']),
              zorder=0,
              transform=ccrs.PlateCarree(), cmap=ListedColormap(['xkcd:pale brown']))

axWS.coastlines(lw=0.5)

axOrog.set_title('Orography anomaly\nIceland - no Iceland\n($m$)', size='medium')
axMLD.set_title('Mixed layer depth anomaly\nIceland - no Iceland\n($hPa$)', size='medium')
axWS.set_title('Wind stress anomaly\nIceland - no Iceland\n($m.s^{-1}$)', size='medium')

In [ ]:
fig.savefig(f"{output_folder}/iceland.png", bbox_extra_artists=(), bbox_inches='tight', format='png')

## Figure - Dynamics expanded

In [ ]:
filt = util.ButterLowPass(order=1, fc=2*10**-3, fs=1, mult=2)

In [ ]:
amoc = {}

for expt in database.keys():
# for expt in ['xoupa', 'xqctb', 'xqctc']:
    print(f"Loading {expt}...")
    amoc[expt] = xr.open_dataset(
        f"{database[expt]}/time_series/{expt}.merid.annual.nc").Merid_Atlantic.sel(
        latitude=26.5, method='nearest').max('depth')

In [ ]:
aabw = {}

for expt in database.keys():
    ts = xr.open_dataset(f"{database[expt]}/time_series/{expt}.merid.annual.nc")
    aabw[expt] = ts.Merid_Atlantic.sel(latitude=-33, method='nearest').min('depth').sortby('t')


In [ ]:
salinity_means = {}
for expt in database.keys():
    salinity_means[expt] = xr.open_dataset(
        f"{database[expt]}/salinity_budgets/{expt}.salinity_means.decadal.nc", cache=False).salinity_means

In [ ]:
salinity_clusters = {}

for expt in ['xpfjb', 'xpfjc', 'xpfjd', 'xpfje', 'xqctc']: 
    salinity_clusters[expt] = xr.open_dataset(
        f"{database[expt]}/salinity_budgets/{expt}.salinity_clusters.decadal.nc").salinity_cluster

In [ ]:
N2 = {}

for expt in ['xpfjb', 'xpfjc', 'xpfjd', 'xpfje', 'xqctc']: 
    ocnd = xr.open_dataset(f"{database[expt]}/na_variables/{expt}.oceandenspg.zone_na.annual.nc").density

    N2[expt] = {}

    for zone in masks_na.keys():
        N2[expt][zone] = (-9.81/1025*ocnd.sel(zone=zone).sel(
            depth_1=slice(0,1000)).diff(dim='depth_1', label='lower')\
                    /ocnd.sel(zone=zone).sel(
            depth_1=slice(0,1000)).depth_1.diff(dim='depth_1', label='lower')).sum('depth_1')

In [ ]:
mld_zone, mldf_zone = {}, {}

for expt in ['xpfjb', 'xpfjc', 'xpfjd', 'xpfje', 'xqctc']: 
    print(f"Loading {expt} MLD...")
    mld_zone[expt], mldf_zone[expt] = {}, {}
    temp_mld = xr.open_dataset(
                f"{database[expt]}/na_variables/{expt}.mld.zone_na.annual.nc").mld
    
    for zone in masks_na.keys():
        mld_zone[expt][zone] = temp_mld.sel(zone=zone)[1:]
        mldf_zone[expt][zone] = filt.process(mld_zone[expt][zone].values)

In [ ]:
fig = plt.figure(figsize=(20, 12), dpi=200)

subfigs = fig.subfigures(1, 5,  wspace=0.1, facecolor='None')

axEXPT, axAMOC, axN2, axS, axSMean, axMLD= {}, {}, {}, {}, {}, {}

iterator = count(0)
experiments = ['xpfjd', 'xpfje', 'xpfjb', 'xpfjc', 'xqctc']


# Grids
for expt in experiments:
    i = next(iterator)

    grid = subfigs[i].add_gridspec(6, 1, height_ratios=[2,6,5,5,5,12], hspace=0)
    
    axEXPT[expt] = subfigs[i].add_subplot(grid[0])
    axAMOC[expt] = subfigs[i].add_subplot(grid[1], facecolor='None')
    axN2[expt] = subfigs[i].add_subplot(grid[2], sharex=axAMOC[expt], facecolor='None')
    axS[expt] = subfigs[i].add_subplot(grid[3], sharex=axAMOC[expt], facecolor='None')
    axSMean[expt] = subfigs[i].add_subplot(grid[4], sharex=axAMOC[expt], facecolor='None')
    axMLD[expt] = subfigs[i].add_subplot(grid[5], facecolor='None')


# Experiment Box

    ax_box = axEXPT[expt].get_position()
    axEXPT[expt].set_visible(False)

    cax = subfigs[i].add_axes([ax_box.x0, ax_box.y1,
                        ax_box.width, 
                        ax_box.height])
    
    cax.get_xaxis().set_visible(False)
    cax.get_yaxis().set_visible(False)
    for loc in ['left', 'right', 'bottom', 'top']: cax.spines[loc].set_visible(False)
    cax.set_facecolor(color_expt[expt])

    at = AnchoredText(name_expt[expt], loc=10, frameon=False,
                      prop=dict(backgroundcolor=color_expt[expt],
                                size='large', color='white'))
    cax.add_artist(at)

axTS = np.array([list(axAMOC.values()), list(axN2.values()), list(axS.values()), list(axSMean.values())]).flat


def plot_LineCollection(ax, t, ts, alpha):
    points = np.array([t, ts]).T.reshape(-1,1,2)
    segments = np.concatenate([points[:-1],points[1:]], axis=1)
    lc = LineCollection(segments, cmap=scm.cm.roma, linewidth=1, alpha=alpha)
    lc.set_array(np.arange(len(ts)))

    ax.add_collection(lc)


# AMOC
for expt in experiments:
    plot_LineCollection(axAMOC[expt], amoc[expt].t.dt.year - start_expt[expt], amoc[expt], 0.03)
    plot_LineCollection(axAMOC[expt], amoc[expt].t.dt.year - start_expt[expt], util.rmean(amoc[expt],30) , 1)

    
# Plot N2 index
for expt in experiments:
    for zone in ['gin', 'irm', 'eur']:
        axN2[expt].plot(N2[expt][zone].t.dt.year - start_expt[expt], util.rmean(N2[expt][zone],30)*1e4,
                  color=color_zones[zone], label=label_zones[zone])


# Plot Salinity budgets
for expt in experiments:
    for zone in ['na', 'tpa', 'pac', 'arc']:
        salinity_anomaly = (salinity_clusters[expt].sel(zone=zone).sel(depth='tot') - salinity_clusters[expt].sel(zone=zone).sel(depth='tot')[0])
        salinity_anomaly -= util.rmean((salinity_clusters[expt].sel(zone=zone).sel(depth='tot') - salinity_clusters[expt].sel(zone=zone).sel(depth='tot')[0]), n=100)
        axS[expt].plot(salinity_clusters[expt].sel(zone=zone).sel(depth='tot').t.dt.year \
                            - start_expt[expt],
                   salinity_anomaly*1e-15, 
                   color=color_zones[zone], label=label_zones[zone])

# Means
for expt in experiments:
    for zone in ['na', 'tpa', 'pac', 'arc']:
        axSMean[expt].plot(salinity_means[expt].sel(zone=zone).sel(depth='tot').t.dt.year \
                            - start_expt[expt],
                   (salinity_means[expt].sel(zone=zone).sel(depth='tot') \
                    - salinity_means[expt].sel(zone=zone).sel(depth='tot')[0]), 
                   color=color_zones[zone], label=label_zones[zone])

# Phases
for expt in experiments:
    axMLD[expt].scatter(mldf_zone[expt]['gin'], mldf_zone[expt]['irm'], 
                               c =scm.cm.roma(np.linspace(0, 1, len(mldf_zone[expt]['gin']))),
                               marker='+', alpha=0.1, zorder=2)


# General parameters
for ax in axTS:
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.grid(which='both', color='lightgrey', linestyle='--', linewidth=0.2, zorder=0)


# Paramters time series
for expt in experiments:
    axAMOC[expt].xaxis.set(ticks_position='top', label_position='top')
    axAMOC[expt].set_xlim([-21500,-13000])
    axAMOC[expt].set_ylim([0,25])
    axAMOC[expt].set_xlabel('Years')
    for loc in ['right', 'bottom']: axAMOC[expt].spines[loc].set_visible(False)

    axN2[expt].tick_params(axis='x', colors='None', which='both')
    axN2[expt].set_ylim([1,-25])
    axN2[expt].yaxis.set(ticks_position='right', label_position='right')
    for loc in ['left', 'top', 'bottom']: axN2[expt].spines[loc].set_visible(False)
    
    axS[expt].tick_params(axis='x', colors='None', which='both')
    axS[expt].axhline(0, color='black', lw=1, linestyle='--')
    for loc in ['right', 'top', 'bottom']: axS[expt].spines[loc].set_visible(False)
    
    axSMean[expt].tick_params(axis='x', colors='None', which='both')
    axSMean[expt].set_ylim([-1,0.3])
    axSMean[expt].yaxis.set(ticks_position='right', label_position='right')
    for loc in ['left', 'top']: axSMean[expt].spines[loc].set_visible(False)

axAMOC['xpfjd'].set_ylabel('AMOC index\nSv')
axN2['xqctc'].set_ylabel(r"$N^2$ Stratification index""\n"r"$e^{-4}s^{-2}$")
axS['xpfjd'].set_ylabel('Salt content - 100-year running mean\ne15 kg')
axSMean['xqctc'].set_ylabel('Salinity means\ng/kg')

axS['xpfjc'].legend(fontsize='xx-small', frameon=False)

for expt in experiments:
    axMLD[expt].set_xlim([10,110])
    axMLD[expt].set_ylim([20,110])
    axMLD[expt].set_xlabel('Mixed layer depth\nin the GIN seas (m)')
    axMLD[expt].grid(which='both', color='lightgrey', linestyle='--', linewidth=0.2, zorder=0)

axMLD['xqctc'].set_ylabel('Mixed layer depth\nin the Irminger Sea (m)')
axMLD['xqctc'].yaxis.set(ticks_position='right', label_position='right')


# # Annotations
i = count(0)
for ax in axAMOC, axN2, axS, axSMean, axMLD:
    for expt in axEXPT.keys():
        txt = ax[expt].annotate(alc[next(i)], xy=(0.93,0.04), xycoords='axes fraction', size=12, weight='bold')
        

In [ ]:
fig.savefig(f"{output_folder}/dynamics_expanded.png", bbox_extra_artists=(), bbox_inches='tight', format='png')

## Figure - Meltwater correlations

In [ ]:
eml_mw = {}
eml_mw['GLAC-1D'] = xr.open_dataset(f"{data_folder}//mw_bins/eml_mw_roll.GLAC-1D.nc")
eml_mw['ICE6G'] = xr.open_dataset(f"{data_folder}/mw_bins/eml_mw_roll.ICE6G.nc")

lreg = {}
indexes = {'GLAC-1D':{}, 'ICE6G':{}}


for zone in ['arc', 'gin', 'elwg']:
    lreg[zone] = stats.linregress(eml_mw['GLAC-1D']['tot'].dropna(dim='year'), 
                                        eml_mw['GLAC-1D'][zone].dropna(dim='year'))
x = np.linspace(eml_mw['GLAC-1D']['tot'].min().values[()], eml_mw['GLAC-1D']['tot'].max().values[()], 10) 

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6,6), dpi=300)


for zone in ['arc', 'gin', 'elwg']:
    ax.scatter(eml_mw['GLAC-1D']['tot'], eml_mw['GLAC-1D'][zone], 
				color=color_fluxes[zone], marker='.', s=10, edgecolors='None', alpha=0.5, linestyle='None',
				label=f'GLAC-1D {zone}')
    ax.plot(x, lreg[zone].slope*x, color=color_fluxes[zone], ls='--')

y_note = {'arc':0.9,'gin':0.85,'elwg':0.8}
for zone in ['arc', 'gin', 'elwg']:
    ax.annotate(label_fluxes[zone]+r' ($r^2$='+f"{lreg[zone].rvalue**2:.02f})",
                xy=(0.1,y_note[zone]), xycoords='axes fraction', size='small', va='top',
                color=color_fluxes[zone])


for loc in ['top', 'right']: ax.spines[loc].set_visible(False)
ax.set_xlabel('Meltwater discharge in region ($Sv$)')
ax.set_ylabel('Total meltwater discharge ($Sv$)')
# ax.legend(fontsize='small', frameon=False)


In [ ]:
fig.savefig(f"{output_folder}/mw_correlation.png", bbox_extra_artists=(), bbox_inches='tight', format='png')

## Figure - Window of oportunity in GLAC1D simulations

In [ ]:
filt = util.ButterLowPass(order=1, fc=2*10**-3, fs=1, mult=2)

mld_zone, mldf_zone = {}, {}

for expt in ['xpfjd', 'xpfje']:
    print(f"Loading {expt} MLD...")
    mld_zone[expt], mldf_zone[expt] = {}, {}
    temp_mld = xr.open_dataset(
                f"{database[expt]}/na_variables/{expt}.mld.zone_na.annual.nc").mld

    for zone in masks_na.keys():
        mld_zone[expt][zone] = temp_mld.sel(zone=zone)[1:]
        mldf_zone[expt][zone] = {}
        mldf_zone[expt][zone]['GLAC_22'] = filt.process(mld_zone[expt][zone].isel(t=slice(0,6500)).values)
        mldf_zone[expt][zone]['GLAC_155'] = filt.process(mld_zone[expt][zone].isel(t=slice(6500,8000)).values)        
    
expt = 'xqctc'
print(f"Loading {expt} MLD...")
mld_zone[expt], mldf_zone[expt] = {}, {}
temp_mld = xr.open_dataset(
            f"{database[expt]}/na_variables/{expt}.mld.zone_na.annual.nc").mld
for zone in masks_na.keys():
        mld_zone[expt][zone] = temp_mld.sel(zone=zone)[1:]
        mldf_zone[expt][zone] = filt.process(mld_zone[expt][zone].values)

In [ ]:
eml_amoc = {}
for expt in ['xpfjb', 'xpfjc', 'xpfjd', 'xpfje', 'GLAC-1D', 'ICE6G', 'xqctc']:
    eml_amoc[expt] = xr.open_dataset(f"{data_folder}/mw_bins/eml_amoc.{expt}.nc").Merid_Atlantic.sortby('year')
    
eml_mw = {}
for expt in ['xpfjb', 'xpfjc', 'xpfjd', 'xpfje', 'GLAC-1D', 'ICE6G', 'xqctc']:
    eml_mw[expt] = xr.open_dataset(f"{data_folder}/mw_bins/eml_mw_roll.{expt}.nc").sortby('year')

In [ ]:
for expt in ['xpfjb', 'xpfjc', 'xpfjd', 'xpfje', 'GLAC-1D', 'ICE6G', 'xqctc']:
    eml_mw[expt]['nh'] = eml_mw[expt]['tot'] - eml_mw[expt]['so'] - eml_mw[expt]['pac']

In [ ]:
fig = plt.figure(figsize=(15, 10), dpi=200)

grid = fig.add_gridspec(2, 3, hspace=0.25, wspace=0.2)

axMLD = {}
axEML = {}
    
axMLD['GLAC_22'] = fig.add_subplot(grid[0,0], facecolor='None')
axMLD['GLAC_155'] = fig.add_subplot(grid[0,1], facecolor='None')
axMLD['GLAC_ts'] = fig.add_subplot(grid[0,2], facecolor='None')

axEML['GLAC_22'] = fig.add_subplot(grid[1,0], facecolor='None')
axEML['GLAC_155'] = fig.add_subplot(grid[1,1], sharex=axEML['GLAC_22'], sharey=axEML['GLAC_22'], facecolor='None')
axEML['GLAC_ts'] = fig.add_subplot(grid[1,2], facecolor='None')


# Phases
axMLD['GLAC_22'].scatter(mldf_zone['xpfjd']['gin']['GLAC_22'], mldf_zone['xpfjd']['irm']['GLAC_22'], 
                            c =color_expt['xpfjd'],
                            marker='.', s=10, edgecolors='None', alpha=0.4, linestyle='None', zorder=2)
axMLD['GLAC_22'].scatter(mldf_zone['xpfje']['gin']['GLAC_22'], mldf_zone['xpfje']['irm']['GLAC_22'], 
                            c =color_expt['xpfje'],
                            marker='.', s=10, edgecolors='None', alpha=0.4, linestyle='None', zorder=2)
axMLD['GLAC_155'].scatter(mldf_zone['xpfjd']['gin']['GLAC_155'], mldf_zone['xpfjd']['irm']['GLAC_155'], 
                          c =color_expt['xpfjd'],
                          marker='.', s=10, edgecolors='None', alpha=0.4, linestyle='None', zorder=2)
axMLD['GLAC_155'].scatter(mldf_zone['xpfje']['gin']['GLAC_155'], mldf_zone['xpfje']['irm']['GLAC_155'], 
                          c =color_expt['xpfje'],
                          marker='.', s=10, edgecolors='None', alpha=0.4, linestyle='None', zorder=2)
axMLD['GLAC_ts'].scatter(mldf_zone['xqctc']['gin'], mldf_zone['xqctc']['irm'], 
                          c =color_expt['xqctc'],
                          marker='.', s=10, edgecolors='None', alpha=0.4, linestyle='None', zorder=2)


for expt in ['GLAC_22', 'GLAC_155', 'GLAC_ts']:
    axMLD[expt].set_xlim([10,110])
    axMLD[expt].set_ylim([20,110])
    axMLD[expt].set_xlabel('Mixed layer depth\nin the GIN seas (m)')
    axMLD[expt].grid(which='both', color='lightgrey', linestyle='--', linewidth=0.2, zorder=0)
    # axMLD[expt].xaxis.set(ticks_position='top', label_position='top')

axMLD['GLAC_22'].set_ylabel('Mixed layer depth\nin the Irminger Sea (m)')

axMLD['GLAC_155'].tick_params(axis='y', colors='None', which='both')

axMLD['GLAC_ts'].set_ylabel('Mixed layer depth\nin the Irminger Sea (m)')
axMLD['GLAC_ts'].yaxis.set(ticks_position='right', label_position='right')

axMLD['GLAC_22'].set_title("GLAC1D meltwater simulations\n21.5 ka BP to 15.5 ka BP", size='medium')
axMLD['GLAC_155'].set_title("GLAC1D meltwater simulations\n15.5 ka BP to 13 ka BP", size='medium')
axMLD['GLAC_ts'].set_title("GLAC1D_ts\n21.5 ka BP to 13 ka BP", size='medium')

def _plot_expts(ax, expts, time_slice):
    for expt in expts:
        ax.plot(eml_mw[expt]['nh'].sel(year=time_slice), util.rmean(eml_amoc[expt].sel(year=time_slice),30), color=color_expt[expt], alpha=0.5, lw=1)
        ax.plot(eml_mw[expt]['nh'].sel(year=time_slice), eml_amoc[expt].sel(year=time_slice), color=color_expt[expt], marker='.', alpha=0.02, linestyle='None')

# Plot experiments states
_plot_expts(axEML['GLAC_22'], ['xpfjd', 'xpfje'], time_slice=slice(-22000,-15500))
_plot_expts(axEML['GLAC_155'], ['xpfjd', 'xpfje'], time_slice=slice(-15500,-13000))
_plot_expts(axEML['GLAC_ts'], ['xqctc'], time_slice=slice(-21500,-13000))
        
# Annotate fits
for expt in ['GLAC_22', 'GLAC_155', 'GLAC_ts']:
    axEML[expt].set_xlim([0,0.35])
    axEML[expt].set_ylim([0,26])
    axEML[expt].axvspan(0, 0.05, alpha=0.05, color=color_spans['merid'], linewidth=0, hatch='//', edgecolor=color_spans['merid'])
    axEML[expt].axvspan(0.05, 0.15, alpha=0.3, color=color_spans['zonal'], linewidth=0)
    axEML[expt].axvspan(0.15, 0.35, alpha=0.05, color=color_spans['cold'], linewidth=0, hatch='//', edgecolor=color_spans['cold'])

xlim = axEML['GLAC_22'].get_xlim()
x_frac = (0.1 - xlim[0]) / (xlim[1] - xlim[0])
axEML['GLAC_22'].annotate("Window of opportunity", xy=(x_frac, 1.00), xycoords='axes fraction',
                          ha='center', va='bottom', fontsize='small', color=color_spans['zonal'])

# Parameters for mw panel
axEML['GLAC_22'].set_xlabel("Total discharge (Sv)", fontsize='large')
axEML['GLAC_22'].set_ylabel("AMOC index (Sv)", fontsize='large')
for loc in ['right', 'top']:
    axEML['GLAC_22'].spines[loc].set_visible(False)

axEML['GLAC_155'].set_xlabel("Total discharge (Sv)", fontsize='large')
for loc in ['right', 'top', 'left']:
    axEML['GLAC_155'].spines[loc].set_visible(False)
axEML['GLAC_155'].tick_params(axis='y', colors='None', which='both')

axEML['GLAC_ts'].set_xlabel("Total discharge (Sv)", fontsize='large')
axEML['GLAC_ts'].set_ylabel("AMOC index (Sv)", fontsize='large')
for loc in ['left', 'top']:
    axEML['GLAC_ts'].spines[loc].set_visible(False)
axEML['GLAC_ts'].yaxis.set_label_position('right')
axEML['GLAC_ts'].yaxis.set_ticks_position('right')

axMLD['GLAC_22'].annotate('a', xy=(0.9,0.95), xycoords='axes fraction', size=12, weight='bold')
axMLD['GLAC_155'].annotate('b', xy=(0.9,0.95), xycoords='axes fraction', size=12, weight='bold')
axMLD['GLAC_ts'].annotate('c', xy=(0.9,0.95), xycoords='axes fraction', size=12, weight='bold')

axEML['GLAC_22'].annotate('d', xy=(0.9,0.95), xycoords='axes fraction', size=12, weight='bold')
axEML['GLAC_155'].annotate('e', xy=(0.9,0.95), xycoords='axes fraction', size=12, weight='bold')
axEML['GLAC_ts'].annotate('f', xy=(0.9,0.95), xycoords='axes fraction', size=12, weight='bold')


In [ ]:
fig.savefig(f"{output_folder}/woo_mw.png", bbox_extra_artists=(), bbox_inches='tight', format='png')

## Figure - Proxy NA updated

In [ ]:
snoll23_folder = f"{home_folder}/work/data/snoll_egu_23"

### Model results

In [ ]:
t_glac = {'BA':((100_000-14_500, 100_000-14_100), (100_000-13_900, 100_000-13_800)),
		  'DO':((100_000-18_200, 100_000-17_850), (100_000-17_700, 100_000-16_700))}

t_ice6g = {'BA':((100_000-14_400, 100_000-13_800), (100_000-17_000, 100_000-16_500)),
		   'DO':((100_000-14_400, 100_000-13_800), (100_000-17_000, 100_000-16_500))}

t_glacts = {'DG':((100_000-21_500, 100_000-21_000), (100_000-14_000, 100_000-13_000))}

In [ ]:
ngrip = {}

for expt in ['xpfjb', 'xpfjd', 'xqctc']:
    ngrip[expt] = xr.open_dataset(
        f"{database[expt]}/perso/{expt}.temp2m.annual.nc").temp_mm_1_5m.sel(
        longitude=318, method='nearest').sel(latitude=75, method='nearest') - 273.15


In [ ]:
means = {'BA':{'sst':{}, 'sat':{}}, 
         'DO':{'sst':{}, 'sat':{}}, 
         'DG':{'sst':{}, 'sat':{}}
         }


# GLAC1D_Mw
expt = 'xpfjd'
temp_sst = xr.open_dataset(f"{database[expt]}/{expt}.oceansurftemppf.annual.nc")
temp_sst = temp_sst.temp_mm_uo.assign_coords(t=temp_sst.t.dt.year)
temp_sat = xr.open_dataset(f"{database[expt]}/{expt}.temp2m.annual.nc")
temp_sat = temp_sat.temp_mm_1_5m.assign_coords(t=temp_sat.t.dt.year)

for period in ['BA', 'DO']:
	means[period]['sst'][expt] = temp_sst.sel(
		t=slice(t_glac[period][1][0], t_glac[period][1][1])).mean('t') - temp_sst.sel(
		t=slice(t_glac[period][0][0], t_glac[period][0][1])).mean('t')
	means[period]['sat'][expt] = temp_sat.sel(
		t=slice(t_glac[period][1][0], t_glac[period][1][1])).mean('t') - temp_sat.sel(
		t=slice(t_glac[period][0][0], t_glac[period][0][1])).mean('t')


# ICE6G_Mw
expt = 'xpfjb'
temp_sst = xr.open_dataset(f"{database[expt]}/{expt}.oceansurftemppf.annual.nc")
temp_sst = temp_sst.temp_mm_uo.assign_coords(t=temp_sst.t.dt.year)
temp_sat = xr.open_dataset(f"{database[expt]}/{expt}.temp2m.annual.nc")
temp_sat = temp_sat.temp_mm_1_5m.assign_coords(t=temp_sat.t.dt.year)

for period in ['BA', 'DO']:
	means[period]['sst'][expt] = temp_sst.sel(
		t=slice(t_ice6g[period][1][0], t_ice6g[period][1][1])).mean('t') - temp_sst.sel(
		t=slice(t_ice6g[period][0][0], t_ice6g[period][0][1])).mean('t')
	means[period]['sat'][expt] = temp_sat.sel(
		t=slice(t_ice6g[period][1][0], t_ice6g[period][1][1])).mean('t') - temp_sat.sel(
		t=slice(t_ice6g[period][0][0], t_ice6g[period][0][1])).mean('t')
    

# GLAC1D_ts
expt = 'xqctc'
temp_sst = xr.open_dataset(f"{database[expt]}/{expt}.oceansurftemppf.annual.nc")
temp_sst = temp_sst.temp_mm_uo.assign_coords(t=temp_sst["t"].dt.year)
temp_sat = xr.open_dataset(f"{database[expt]}/{expt}.temp2m.annual.nc")
temp_sat = temp_sat.temp_mm_1_5m.assign_coords(t=temp_sat["t"].dt.year)

for period in ['DG']:
	means[period]['sst'][expt] = temp_sst.sel(
		t=slice(t_glacts[period][1][0], t_glacts[period][1][1])).mean('t') - temp_sst.sel(
		t=slice(t_glacts[period][0][0], t_glacts[period][0][1])).mean('t')
	means[period]['sat'][expt] = temp_sat.sel(
		t=slice(t_glacts[period][1][0], t_glacts[period][1][1])).mean('t') - temp_sat.sel(
		t=slice(t_glacts[period][0][0], t_glacts[period][0][1])).mean('t')


### Proxies

In [ ]:
t_dg = ((21_500, 19_500), (14_600, 13_000))
t_ba = ((16_000, 15_000), (14_500, 13_500))
t_do = ((39_500, 38_500), (38_000, 37_000))

cold_ba, warm_ba = {}, {}
cold_dg, warm_dg = {}, {}
cold_do, warm_do = {}, {}

proxy_dg = {}
proxy_ba = {}
proxy_do = {}


In [ ]:
### Martin 2023 - Deglaciation NGRIP

proxy_ngrip = pd.read_excel(f"{data_folder}/martin_nature_2023/41586_2023_5875_MOESM3_ESM.xlsx", sheet_name='Fig 1c',
              skiprows=14, header=0, names=['age', 'age_bic', 'temp', 'source'])
# proxy_ngrip = proxy_ngrip.where(proxy_ngrip.age>13_000).dropna()
# proxy_ngrip = proxy_ngrip.where(proxy_ngrip.age<22_000).dropna()

**Shakun et al. 2022 - Deglaciation and BA**

In [ ]:
df = pd.read_excel(f"{snoll23_folder}/proxy_data/shakun_surface_temp/shakun_surface_temp_stack.xls", None)
df

In [ ]:
meta = pd.read_csv(
    f"{snoll23_folder}/proxy_data/shakun_surface_temp/shakun_surface_temp_metadata_sheet.csv", 
    encoding = "ISO-8859-1", usecols=[1,2,4,5]).rename(columns={'Lat (°)':'lat', 'Lon (°)':'lon'})


df = pd.read_excel(f"{snoll23_folder}/proxy_data/shakun_surface_temp/shakun_surface_temp_stack.xls", None)

names = [key for key in list(df.keys())[3:] if key not in ['SU81-18_Waelbroeck']]

meta_proxies = {}

i=count(0)
for name in names:
    meta_proxies[name] = meta.iloc[next(i)]

for site in names:
    df_temp = df[site][1:]
    if 'Published age' in list(df_temp.columns):
        df_temp = df_temp.rename(columns={'Published age':'Age'})
    if 'Published temperature' in list(df_temp.columns):
        df_temp = df_temp.rename(columns={'Published temperature':'Temperature'})

    cold_ba[site] = df_temp.loc[df_temp['Age'].astype(int)<t_ba[0][0]].loc[df_temp['Age'].astype(int)>t_ba[0][1]].dropna(subset='Temperature')['Temperature']
    warm_ba[site] = df_temp.loc[df_temp['Age'].astype(int)<t_ba[1][0]].loc[df_temp['Age'].astype(int)>t_ba[1][1]].dropna(subset='Temperature')['Temperature']
    proxy_ba[site] = warm_ba[site].mean() - cold_ba[site].mean()
    
    cold_dg[site] = df_temp.loc[df_temp['Age'].astype(int)<t_dg[0][0]].loc[df_temp['Age'].astype(int)>t_dg[0][1]].dropna(subset='Temperature')['Temperature']
    warm_dg[site] = df_temp.loc[df_temp['Age'].astype(int)<t_dg[1][0]].loc[df_temp['Age'].astype(int)>t_dg[1][1]].dropna(subset='Temperature')['Temperature']
    proxy_dg[site] = warm_dg[site].mean() - cold_dg[site].mean()

**Eldevik et al. 2014**

In [ ]:
eldevik_proxies = {'MD95-2010':(4.566160,66.684167), 'MD99-2284':(-0.980167,62.374667)}

for site in eldevik_proxies.keys():
    
    df_temp = pd.read_csv(f"{data_folder}/eldevik_qsr_22/dokken_{site}.tab", 
                          sep="\t", skiprows=41, header=0, usecols=[0,1,29,30], names=['Depth', 'Age', 'SST', 'Anomaly'])

    cold_ba[site] = df_temp.loc[(df_temp['Age']*1000).astype(int)<t_ba[0][0]].loc[(df_temp['Age']*1000).astype(int)>t_ba[0][1]].dropna(subset='SST')['SST']
    warm_ba[site] = df_temp.loc[(df_temp['Age']*1000).astype(int)<t_ba[1][0]].loc[(df_temp['Age']*1000).astype(int)>t_ba[1][1]].dropna(subset='SST')['SST']
    proxy_ba[site] = warm_ba[site].mean() - cold_ba[site].mean()
    
    cold_dg[site] = df_temp.loc[(df_temp['Age']*1000).astype(int)<t_dg[0][0]].loc[(df_temp['Age']*1000).astype(int)>t_dg[0][1]].dropna(subset='SST')['SST']
    warm_dg[site] = df_temp.loc[(df_temp['Age']*1000).astype(int)<t_dg[1][0]].loc[(df_temp['Age']*1000).astype(int)>t_dg[1][1]].dropna(subset='SST')['SST']
    proxy_dg[site] = warm_dg[site].mean() - cold_dg[site].mean()


**Naafs et al. 2013**

In [ ]:
naafs_proxies = {'U1313B':(-32.957300,41.000023)}

for site in naafs_proxies.keys():
    
    df_temp = pd.read_csv(f"{data_folder}/naafs_paleoceanography_13/naafs_{site}.tab", 
                          sep="\t", skiprows=23, header=0, usecols=[2,5,6], names=['Depth', 'Age', 'SST'])

    cold_ba[site] = df_temp.loc[(df_temp['Age']*1000).astype(int)<t_ba[0][0]].loc[(df_temp['Age']*1000).astype(int)>t_ba[0][1]].dropna(subset='SST')['SST']
    warm_ba[site] = df_temp.loc[(df_temp['Age']*1000).astype(int)<t_ba[1][0]].loc[(df_temp['Age']*1000).astype(int)>t_ba[1][1]].dropna(subset='SST')['SST']
    proxy_ba[site] = warm_ba[site].mean() - cold_ba[site].mean()

    cold_dg[site] = df_temp.loc[(df_temp['Age']*1000).astype(int)<t_dg[0][0]].loc[(df_temp['Age']*1000).astype(int)>t_dg[0][1]].dropna(subset='SST')['SST']
    warm_dg[site] = df_temp.loc[(df_temp['Age']*1000).astype(int)<t_dg[1][0]].loc[(df_temp['Age']*1000).astype(int)>t_dg[1][1]].dropna(subset='SST')['SST']
    proxy_dg[site] = warm_dg[site].mean() - cold_dg[site].mean()
    

**Pedro et al. 2022**

In [ ]:
pedro_proxies = pd.read_csv(f"{data_folder}/pedro_qsr_22/gi8_gs9.txt", 
                            sep="\t", skiprows=8, header=0, names=['lat', 'lon', 'temp', 'std', 't-test'])

for index, row in pedro_proxies.iterrows():
    proxy_do[f"pedro_{index}"] = row.temp

In [ ]:
sst_stacks = pd.read_csv(f"{data_folder}/pedro_qsr_22/stacks/All_stack_proxy.txt", 
                         sep="\t", skiprows=2, header=0, names=['age', 'temp', 'std', 'calbration', 'rms'])

### Plot

In [ ]:
cmaps = {'sst': scm.cm.vik, 'sst_diff':scm.cm.vik,
         'sat': scm.cm.vik, 'sat_diff':scm.cm.vik}

norms = {'sst': TwoSlopeNorm(vmin=-60, vcenter=0, vmax=40), 'sst_diff':TwoSlopeNorm(vmin=-10, vcenter=0, vmax=10),
         'sat': TwoSlopeNorm(vmin=-60, vcenter=0, vmax=40), 'sat_diff':TwoSlopeNorm(vmin=-10, vcenter=0, vmax=10)}


In [ ]:
axTS = {'BA': {}, 'DO': {}, 'DG': {}}
axProxy = {'BA': {}, 'DO': {}, 'DG': {}}

projection_map = ccrs.PlateCarree(central_longitude=-30.0)

fig = plt.figure(figsize=(18, 12), dpi=300)

gs = gridspec.GridSpec(nrows=6, ncols=4,
                       hspace=0.3, height_ratios=[2, 2, 2, 2, 2, 2],
                       wspace=0.1, width_ratios=[2, 3, 3, 3])

axTS['BA']['proxy'] = fig.add_subplot(gs[0, 0], facecolor='None')
axTS['BA']['model'] = fig.add_subplot(gs[1, 0], facecolor='None', sharex=axTS['BA']['proxy'], sharey=axTS['BA']['proxy'])
axProxy['BA']['xpfjd'] = fig.add_subplot(gs[0:2, 1], facecolor='None', projection=projection_map)
axProxy['BA']['xpfjb'] = fig.add_subplot(gs[0:2, 2], facecolor='None', projection=projection_map)

axTS['DO']['proxy'] = fig.add_subplot(gs[2, 0], facecolor='None')
axTS['DO']['model'] = fig.add_subplot(gs[3, 0], facecolor='None', sharey=axTS['DO']['proxy'])
axProxy['DO']['xpfjd'] = fig.add_subplot(gs[2:4, 1], facecolor='None', projection=projection_map)
axProxy['DO']['xpfjb'] = fig.add_subplot(gs[2:4, 2], facecolor='None', projection=projection_map)

axTS['DG']['proxy'] = fig.add_subplot(gs[4, 0], facecolor='None')
axTS['DG']['model'] = fig.add_subplot(gs[5, 0], facecolor='None', sharex=axTS['DG']['proxy'], sharey=axTS['DG']['proxy'])
axProxy['DG']['xqctc'] = fig.add_subplot(gs[4:6, 1], facecolor='None', projection=projection_map)

# -----

t2years_hadcm3 = lambda t: -(t-100_000)/1000
t2years_martin = lambda t: t/1000

def plot_spans_model(ax, years, temp, rmean, tcold, twarm, color, t2years_method, dim=None):
	ax.plot(t2years_method(years), util.rmean(temp, rmean), alpha=0.5, lw=0.5, color=color)
	temp_ts = temp.where(years>=tcold[0]).where(years<=tcold[1]).dropna(dim=dim)
	ax.plot(t2years_method(temp_ts.t.dt.year), util.rmean(temp_ts,rmean), alpha=1, lw=0.5, color=color_spans['cold'], zorder=0)
	ax.plot(t2years_method(temp_ts.t.dt.year), util.rmean(temp_ts,rmean), alpha=0.5, lw=3, color=color, zorder=0)
	temp_ts = temp.where(years>twarm[0]).where(years<twarm[1]).dropna(dim=dim)
	ax.plot(t2years_method(temp_ts.t.dt.year), util.rmean(temp_ts,rmean), alpha=1, lw=0.5, color=color_spans['merid'], zorder=0)
	ax.plot(t2years_method(temp_ts.t.dt.year), util.rmean(temp_ts,rmean), alpha=0.3, lw=3, color=color, zorder=0)

def plot_spans_proxy(ax, proxy, tcold, twarm, color, t2years_method):
	ax.plot(t2years_method(proxy.age), proxy.temp, color='black', alpha=1, lw=0.3, zorder=1)
	temp_ts = proxy.where(proxy.age<=tcold[0]).where(proxy.age>=tcold[1]).dropna()
	ax.plot(t2years_method(temp_ts.age), temp_ts.temp, color=color_spans['cold'], lw=0.5, alpha=1, zorder=0)
	ax.plot(t2years_method(temp_ts.age), temp_ts.temp, color=color, lw=3, alpha=0.5, zorder=0)
	temp_ts = proxy.where(proxy.age<=twarm[0]).where(proxy.age>=twarm[1]).dropna()
	ax.plot(t2years_method(temp_ts.age), temp_ts.temp, color=color_spans['merid'], lw=0.5, alpha=1, zorder=0)
	ax.plot(t2years_method(temp_ts.age), temp_ts.temp, color=color, lw=3, alpha=0.5, zorder=0)

# -----

# Times Series

# BA Warming & DO events
for period in ['BA', 'DO']:
	plot_spans_model(axTS[period]['model'], ngrip['xpfjd'].t.dt.year, ngrip['xpfjd'], 50, t_glac[period][0], t_glac[period][1], color_expt['xpfjd'], t2years_hadcm3, dim='t')
	plot_spans_model(axTS[period]['model'], ngrip['xpfjb'].t.dt.year, ngrip['xpfjb'], 50, t_ice6g[period][0], t_ice6g[period][1], color_expt['xpfjb'], t2years_hadcm3, dim='t')

plot_spans_proxy(axTS['BA']['proxy'], proxy_ngrip, t_ba[0], t_ba[1], 'black', t2years_martin)
plot_spans_proxy(axTS['DO']['proxy'], proxy_ngrip, t_do[0], t_do[1], 'black', t2years_martin)

# Deglaciation
plot_spans_model(axTS['DG']['model'], ngrip['xqctc'].t.dt.year, ngrip['xqctc'], 50, t_glacts['DG'][0], t_glacts['DG'][1], color_expt['xqctc'], t2years_hadcm3, dim='t')
plot_spans_proxy(axTS['DG']['proxy'], proxy_ngrip, t_dg[0], t_dg[1], 'black', t2years_martin)

# -----

# Anomaly maps

for expt in ['xpfjd', 'xpfjb']:
    for period in ['BA', 'DO']:
        axProxy[period][expt].pcolormesh(means[period]['sst'][expt].longitude, means[period]['sst'][expt].latitude,
                                         means[period]['sst'][expt], transform=ccrs.PlateCarree(
        ),
            zorder=1, cmap=cmaps['sst_diff'], norm=norms['sst_diff'])
        cm = axProxy[period][expt].pcolormesh(means[period]['sat'][expt].longitude, means[period]['sat'][expt].latitude,
                                              means[period]['sat'][expt], transform=ccrs.PlateCarree(
        ),
            zorder=0, cmap=cmaps['sat_diff'], norm=norms['sat_diff'])

axProxy['DG']['xqctc'].pcolormesh(means['DG']['sst']['xqctc'].longitude, means['DG']['sst']['xqctc'].latitude,
                                  means['DG']['sst']['xqctc'], transform=ccrs.PlateCarree(
),
    zorder=1, cmap=cmaps['sst_diff'], norm=norms['sst_diff'])
cm = axProxy['DG']['xqctc'].pcolormesh(means['DG']['sat']['xqctc'].longitude, means['DG']['sat']['xqctc'].latitude,
                                       means['DG']['sat']['xqctc'], transform=ccrs.PlateCarree(
),
    zorder=0, cmap=cmaps['sat_diff'], norm=norms['sat_diff'])

# Colorbars
cb = plt.colorbar(cm, cax=make_axes_locatable(axProxy['BA']['xpfjb']).append_axes(
    "right", size="2%", pad=0, axes_class=matplotlib.axes.Axes))
cb.set_label(
    label="Surface (Atmosphere and Sea)\nTemperature anomaly (°C)", size='medium')
cb = plt.colorbar(cm, cax=make_axes_locatable(axProxy['DO']['xpfjb']).append_axes(
    "right", size="2%", pad=0, axes_class=matplotlib.axes.Axes))
cb.set_label(
    label="Surface (Atmosphere and Sea)\nTemperature anomaly (°C)", size='medium')
cb = plt.colorbar(cm, cax=make_axes_locatable(axProxy['DG']['xqctc']).append_axes(
    "right", size="2%", pad=0, axes_class=matplotlib.axes.Axes))
cb.set_label(
    label="Surface (Atmosphere and Sea)\nTemperature anomaly (°C)", size='medium')


# -----

# Proxies

# Shakun22
for expt in ['xpfjd', 'xpfjb']:

    for site in meta_proxies.keys():
        if not np.isnan(proxy_ba[site]):
            axProxy['BA'][expt].scatter(meta_proxies[site].lon, meta_proxies[site].lat,
                                        c=proxy_ba[site], cmap=cmaps['sst_diff'], norm=norms['sst_diff'],
                                        marker='o', zorder=2, s=20, linewidth=0.5, edgecolor='white',
                                        transform=ccrs.PlateCarree())
for site in meta_proxies.keys():
    if not np.isnan(proxy_dg[site]):
        axProxy['DG']['xqctc'].scatter(meta_proxies[site].lon, meta_proxies[site].lat,
                                       c=proxy_dg[site], cmap=cmaps['sst_diff'], norm=norms['sst_diff'],
                                       marker='o', zorder=2, s=20, linewidth=0.5, edgecolor='white',
                                       transform=ccrs.PlateCarree())

# Eldevik24
for expt in ['xpfjd', 'xpfjb']:
    for site in eldevik_proxies.keys():
        if not np.isnan(proxy_ba[site]):
            axProxy['BA'][expt].scatter(eldevik_proxies[site][0], eldevik_proxies[site][1],
                                        c=proxy_ba[site], cmap=cmaps['sst_diff'], norm=norms['sst_diff'],
                                        marker='o', zorder=2, s=20, linewidth=0.5, edgecolor='white',
                                        transform=ccrs.PlateCarree())
for site in eldevik_proxies.keys():
    if not np.isnan(proxy_dg[site]):
        axProxy['DG']['xqctc'].scatter(eldevik_proxies[site][0], eldevik_proxies[site][1],
                                       c=proxy_dg[site], cmap=cmaps['sst_diff'], norm=norms['sst_diff'],
                                       marker='o', zorder=2, s=20, linewidth=0.5, edgecolor='white',
                                       transform=ccrs.PlateCarree())

# naafs13
for expt in ['xpfjd', 'xpfjb']:
    for site in naafs_proxies.keys():
        if not np.isnan(proxy_ba[site]):
            axProxy['BA'][expt].scatter(naafs_proxies[site][0], naafs_proxies[site][1],
                                        c=proxy_ba[site], cmap=cmaps['sst_diff'], norm=norms['sst_diff'],
                                        marker='o', zorder=2, s=20, linewidth=0.5, edgecolor='white',
                                        transform=ccrs.PlateCarree())
for site in naafs_proxies.keys():
    if not np.isnan(proxy_dg[site]):
        axProxy['DG']['xqctc'].scatter(naafs_proxies[site][0], naafs_proxies[site][1],
                                       c=proxy_dg[site], cmap=cmaps['sst_diff'], norm=norms['sst_diff'],
                                       marker='o', zorder=2, s=20, linewidth=0.5, edgecolor='white',
                                       transform=ccrs.PlateCarree())
# Pedro22
for expt in ['xpfjd', 'xpfjb']:
    for index in pedro_proxies.index:
        site = f"pedro_{index}"
        if not np.isnan(proxy_do[site]):
            axProxy['DO'][expt].scatter(pedro_proxies.loc[index, 'lon'], pedro_proxies.loc[index, 'lat'],
                                        c=proxy_do[site], cmap=cmaps['sst_diff'], norm=norms['sst_diff'],
                                        marker='o', zorder=2, s=20, linewidth=0.5, edgecolor='white',
                                        transform=ccrs.PlateCarree())


# -----

# Parameters timeseries
for period in ['BA', 'DO', 'DG']:
    axTS[period]['proxy'].set_ylabel("NGRIP Temperature\n($°C$)")

axTS['BA']['proxy'].set_xlim([17, 13])
axTS['BA']['proxy'].set_ylim([-52, -33])
for loc in ['right', 'bottom']:
    axTS['BA']['proxy'].spines[loc].set_visible(False)
axTS['BA']['proxy'].set_xlabel("Years ($ka BP$)")
axTS['BA']['proxy'].xaxis.set(ticks_position='top', label_position='top')
for loc in ['right', 'top', 'bottom']:
    axTS['BA']['model'].spines[loc].set_visible(False)
axTS['BA']['model'].tick_params(axis='x', colors='None', which='both')

axTS['DO']['proxy'].set_xlim([40, 36])
axTS['DO']['proxy'].set_ylim([-52, -33])
for loc in ['right', 'bottom']:
    axTS['DO']['proxy'].spines[loc].set_visible(False)
axTS['DO']['proxy'].set_xlabel("Years ($ka BP$)")
axTS['DO']['proxy'].xaxis.set(ticks_position='top', label_position='top')
axTS['DO']['model'].set_xlim([18.5, 13.5])
for loc in ['right', 'top']:
    axTS['DO']['model'].spines[loc].set_visible(False)
axTS['DO']['model'].set_xlabel("Years ($ka BP$)")

axTS['DG']['proxy'].set_xlim([21.5, 13])
axTS['DG']['proxy'].set_ylim([-52, -33])
for loc in ['right', 'top', 'bottom']:
    axTS['DG']['proxy'].spines[loc].set_visible(False)
axTS['DG']['proxy'].tick_params(axis='x', colors='None', which='both')
for loc in ['right', 'top']:
    axTS['DG']['model'].spines[loc].set_visible(False)
axTS['DO']['model'].set_xlabel("Years ($ka BP$)")


# Parameters maps
for expt in ['xpfjb', 'xpfjd']:
    for period in ['BA', 'DO']:
        axProxy[period][expt].set_extent([280, 380, 25, 85])
        axProxy[period][expt].coastlines()

# axProxy['DG']['xqctc'].set_extent([280, 380, 25, 85])
axProxy['DG']['xqctc'].coastlines()

# Annotations
axProxy['BA']['xpfjb'].set_title("ICE6G_Mw\nBølling–Allerød warming")
axProxy['BA']['xpfjd'].set_title("GLAC-1D_Mw\nBølling–Allerød warming")

axProxy['DO']['xpfjb'].set_title("ICE6G_Mw\nDansgaard-Oeschger warming")
axProxy['DO']['xpfjd'].set_title("GLAC-1D_Mw\nDansgaard-Oeschger warming")

axProxy['DG']['xqctc'].set_title("GLAC-1D_ts\nDeglaciation warming")

In [ ]:
fig.savefig(f"{output_folder}/proxies.png", bbox_extra_artists=(), bbox_inches='tight', format='png')